In [ ]:
#!/usr/bin/env python3
"""
prior_guided_pca.py
===================
Methods:
  A.  Global TruncSVD (no centering)
  A*. Global TruncSVD (centered)
  A2. Global TruncSVD (SNV-Pre, no centering)
  A2*.Global TruncSVD (SNV-Pre, centered)
  A3. Global TruncSVD (MSC-Pre, no centering)
  A3*.Global TruncSVD (MSC-Pre, centered)
  B2. Raw Peak-Only
  C2. Raw + Comp TruncSVD
  D.  Rect + Comp TruncSVD
  E.  Local + Comp TruncSVD
  F.  PLS-DA (baseline)

Experiments (CSV default):
  1. Stratified 5-fold learning curves (mean ± std)
  2. Water OOD: ID=Clear, OOD={Turbid,Foamy} + simulated perturbations on Turbid

Metrics: OA, AA, macro-F1, Cohen κ

Data:
  Default: CSV reflectance library (350–2500 nm, cropped to 940–1680 nm)
  Legacy : SWIR .mat cube pipeline is kept for reference (main_swir)
"""

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

try:
    import scipy.io as sio

    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    import h5py

    HAS_H5PY = True
except ImportError:
    HAS_H5PY = False


def _loadmat_any(mat_path):
    """
    Universal .mat loader — handles ALL MATLAB versions:
      v4 / v6 / v7   → scipy.io.loadmat  (existing behaviour)
      v7.3 (HDF5)    → h5py              (fixes NotImplementedError)

    MATLAB v7.3 files are HDF5 archives.  MATLAB stores arrays in
    column-major (Fortran) order, so the dimensions appear *reversed*
    when read by h5py.  This wrapper transposes every array so that
    the returned dict has the same layout as scipy.io.loadmat would.

    Parameters
    ----------
    mat_path : str or Path

    Returns
    -------
    dict  {variable_name: numpy_array, ...}
    """
    mat_path = str(mat_path)

    # ── Try scipy first (v4 / v6 / v7) ──
    if HAS_SCIPY:
        try:
            return sio.loadmat(mat_path)
        except NotImplementedError:
            pass  # v7.3 HDF5 — fall through to h5py

    # ── h5py path for MATLAB v7.3 ──
    if not HAS_H5PY:
        raise ImportError(
            "MATLAB v7.3 (.mat) files require h5py.\n"
            "Install it with:  pip install h5py"
        )

    import numpy as np

    def _read_item(obj):
        """Recursively convert h5py Dataset/Group → numpy array."""
        if isinstance(obj, h5py.Dataset):
            arr = obj[()]
            # h5py reads MATLAB arrays with axes reversed (column-major → row-major).
            # Transpose to restore the original MATLAB axis order so the rest of
            # the pipeline (which expects cube shape (H, W, B) etc.) keeps working.
            if arr.ndim >= 2:
                arr = arr.T
            return arr
        if isinstance(obj, h5py.Group):
            return {k: _read_item(v) for k, v in obj.items()}
        return obj

    out = {}
    with h5py.File(mat_path, "r") as f:
        for k in f.keys():
            if not k.startswith("#"):  # skip HDF5 metadata keys
                out[k] = _read_item(f[k])
    return out


from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             cohen_kappa_score, balanced_accuracy_score,
                             classification_report)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.cross_decomposition import PLSRegression

# ─── Spectral Configuration — SWIR Plastics ───────────────────────────────────
# Sensor: SWIR pushbroom, 900-1700 nm, 81 bands raw.
# Published .mat uses 60 "clean" bands: 940-1340 nm + 1460-1680 nm.
# Water vapour absorption window (1340-1460 nm) removed by dataset authors.
WAVELENGTHS_SWIR = np.concatenate([
    np.linspace(940, 1340, 40),  # 40 bands, C-H overtone / combination region
    np.linspace(1460, 1680, 20),  # 20 bands, C-H 1st overtone region
])  # shape: (60,)

# ⚠  The exact wavelength vector should be read from the .mat if available:
#    wvl = sio.loadmat(f)["wavelength"].ravel()
#    WAVELENGTHS_SWIR = wvl
# The array above is an approximation for when the .mat has no wavelength field.

# Plastic SWIR absorption peaks (literature, independent of this dataset)
# Sources: Workman & Weyer (2012), Garaba & Dierssen (2020)
#   1210 nm — C-H 2nd overtone  (PE, PP)
#   1390 nm — C-H combination   (PP, PS)
#   1490 nm — C-H combination   (LDPE, PET)
#   1660 nm — C-H 1st overtone  (PE, PP, PS)
PRIOR_PEAKS_NM = [1130, 1210,1390,1490, 1660]
SIGMA_NM = 10  # Gaussian half-width ≈ 1 instrument resolution unit

WAVELENGTHS = WAVELENGTHS_SWIR  # active wavelength array (can be swapped)
N_DIM = 20  # target feature dims (60 bands → 20 is reasonable)


def nm_to_idx(wavelengths, peak_nm):
    return [int(np.argmin(np.abs(wavelengths - p))) for p in peak_nm]


def _compute_sigma_bands(wavelengths, sigma_nm):
    spacing = (wavelengths[-1] - wavelengths[0]) / (len(wavelengths) - 1)
    return max(1, int(round(sigma_nm / spacing)))


PRIOR_PEAKS_IDX = nm_to_idx(WAVELENGTHS, PRIOR_PEAKS_NM)
SIGMA_BANDS = _compute_sigma_bands(WAVELENGTHS, SIGMA_NM)


# ─── 1. Data Loaders ──────────────────────────────────────────────────────────

def probe_mat_keys(mat_path):
    """Print variable names and shapes inside a .mat file."""
    mat = _loadmat_any(mat_path)
    print(f"\n── {Path(mat_path).name} ──")
    for k, v in mat.items():
        if not k.startswith("__"):
            shape = v.shape if hasattr(v, "shape") else type(v)
            print(f"  {k:20s}  {shape}")


def discover_swir_cubes(data_dir="swir_cubes",
                        mask_dir=None,
                        mat_files=None):
    """
    只收集 cube / mask 的路径信息，不真正加载光谱数据。
    用于 lazy LOCO：后续按 fold 再逐个读取。
    """
    data_dir = Path(data_dir)
    mask_dir = Path(mask_dir) if mask_dir is not None else data_dir

    if mat_files is None:
        mat_files = sorted(data_dir.glob("cube*.mat"))
    else:
        mat_files = [data_dir / f for f in mat_files]

    if not mat_files:
        raise FileNotFoundError(f"No cube*.mat files found in {data_dir}")

    cube_infos = []
    for f in mat_files:
        stem = f.stem
        mask_stem = stem.replace("cube", "masks", 1)
        bmp_folder = mask_dir / mask_stem
        mat_mask_f = mask_dir / (mask_stem + ".mat")

        if bmp_folder.exists() and bmp_folder.is_dir():
            mp, bmp_dir = None, bmp_folder
        elif mat_mask_f.exists():
            mp, bmp_dir = mat_mask_f, None
        else:
            mp, bmp_dir = None, None
            print(f" [warn] No separate mask found for {f.name}; trying inside cube.")

        cube_infos.append({
            "name": stem,
            "mat_path": f,
            "mask_path": mp,
            "mask_bmp_folder": bmp_dir,
        })

    return cube_infos


def load_mask_folder(mask_folder, max_class_id=9, mask_threshold=128):
    """
    Build a single integer label mask from a folder of per-class JPG files.

    Dataset convention (Balsi / Moroni / Bouchelaghem):
        01_PET.jpg, 02_PE.jpg, 03_PVC.jpg, 04_white.jpg, 05_PP.jpg,
        06_PS.jpg,  07_mixed.jpg, 08_nonplastic.jpg, 09_background.jpg
        10+  = composite masks (10_all_materials etc.) — SKIPPED

    Each JPG is a per-class binary map (but JPG compression may introduce intermediate
    gray values on edges). Use a mid threshold to avoid labeling compression artifacts.

    Parameters
    ----------
    mask_folder  : str or Path — folder containing the JPG files
    max_class_id : int — only load files whose leading number <= this value.
                   Default 9 skips composite masks (10_all_materials etc.)
    mask_threshold : int — pixel >= threshold is treated as belonging to the class.
                     Default 128 is robust to JPG compression artifacts.

    Returns
    -------
    mask   : (H, W) int32 array
             0  = unlabeled / background (exclude with exclude_bg_label=0)
             1  = PET, 2 = PE, 3 = PVC, 4 = white, 5 = PP,
             6  = PS,  7 = mixed, 8 = non-plastic, 9 = background
    names  : dict {class_int → filename_stem}  for reference

    Notes
    -----
    Requires Pillow:  pip install Pillow
    If a pixel appears in multiple JPG files the LAST (highest-numbered)
    class written wins.  For this dataset classes are mutually exclusive
    so this should not matter.
    """
    import re
    try:
        from PIL import Image
    except ImportError:
        raise ImportError("JPG masks require Pillow.  Install with:  pip install Pillow")

    mask_folder = Path(mask_folder)
    pattern = re.compile(r"^(\d+)_")

    # Collect only single-class BMP files (leading number <= max_class_id)
    bmp_files = []
    for f in sorted(mask_folder.glob("*.jpg")):
        m = pattern.match(f.stem)
        if m and int(m.group(1)) <= max_class_id:
            bmp_files.append((int(m.group(1)), f))

    if not bmp_files:
        raise FileNotFoundError(
            f"No JPG files with leading number <= {max_class_id} found in {mask_folder}.\n"
            f"Files present: {[p.name for p in mask_folder.glob('*.jpg')]}"
        )

    # Read first BMP to get spatial dimensions
    img0 = np.array(Image.open(bmp_files[0][1]).convert("L"))
    H, W = img0.shape
    mask = np.zeros((H, W), dtype=np.int32)
    names = {}

    for cls_id, f in bmp_files:
        img = np.array(Image.open(f).convert("L"))
        if img.shape != (H, W):
            raise ValueError(
                f"JPG shape mismatch: {f.name} is {img.shape}, "
                f"expected ({H}, {W})."
            )
        thr = mask_threshold
        if int(img.max()) <= thr:
            thr = 1
        mask[img >= thr] = cls_id
        names[cls_id] = f.stem

    print(f"  [JPG masks] loaded {len(bmp_files)} classes from {mask_folder.name}/")
    print(f"  spatial size: {H}×{W}  |  classes: {names}")
    return mask, names


def load_single_cube(mat_path,
                     cube_key="cubo",
                     mask_path=None,
                     mask_bmp_folder=None,
                     mask_key=None,
                     wvl_key=None,
                     exclude_bg_label=9):
    """
    Load one SWIR .mat cube (low‑memory implementation).


    """
    # 先读 .mat，但暂时不复制大数组
    mat_cube = _loadmat_any(mat_path)
    cube_raw = mat_cube[cube_key]  # 原始 dtype, shape (H, W, B)
    H, W, B = cube_raw.shape
    n_pixels = H * W

    # ── Wavelength vector ──
    if wvl_key and wvl_key in mat_cube:
        wvl = mat_cube[wvl_key].ravel().astype(float)
    else:
        wvl = (WAVELENGTHS_SWIR[:B]
               if B <= len(WAVELENGTHS_SWIR) else WAVELENGTHS_SWIR)

    # ── 先加载 mask，再决定要保留哪些像素 ──
    if mask_bmp_folder is not None:
        # BMP per class layout
        raw_mask, _bmp_names = load_mask_folder(mask_bmp_folder)
        if raw_mask.shape != (H, W):
            raise ValueError(
                f"BMP mask spatial size {raw_mask.shape} does not match "
                f"cube spatial size ({H}, {W}).\n"
                f"Check that the mask folder matches this cube."
            )
        mask = raw_mask.ravel()
    else:
        mat_mask = _loadmat_any(mask_path) if mask_path is not None else mat_cube

        if mask_key is not None:
            mask = mat_mask[mask_key].astype(int).ravel()
        else:
            candidates = {
                k: v for k, v in mat_mask.items()
                if not k.startswith("__")
                   and hasattr(v, "shape")
                   and int(np.prod(v.shape)) == n_pixels
                   and k != cube_key
            }
            if not candidates:
                avail = {k: v.shape for k, v in mat_mask.items()
                         if hasattr(v, "shape") and not k.startswith("__")}
                raise KeyError(
                    f"Cannot auto-detect mask key in '{mask_path or mat_path}'\n"
                    f"  cube shape = ({H}, {W}, {B}), need array with {n_pixels} elements.\n"
                    f"  Available arrays: {avail}\n"
                    f"  Tip: pass mask_key='<name>' explicitly."
                )
            mask_key_found, mask_arr = next(iter(candidates.items()))
            print(f"  [auto-detect] mask_key='{mask_key_found}' shape={mask_arr.shape}")
            mask = mask_arr.astype(int).ravel()

    # ── 只保留有效像素的索引，不立刻展开整块 ──
    valid_idx = np.where((mask != 0) & (mask != exclude_bg_label))[0]
    if valid_idx.size == 0:
        raise RuntimeError(f"No valid pixels left in {mat_path} after background removal.")
    mask_val = mask[valid_idx]

    # ── 到这一步，才真正构造 X: 只为选中的像素建矩阵 ──
    # 先将 cube 转成 float32 再 reshape，但只保留 valid_idx 行
    row_indices = valid_idx // W  # 每个有效像素属于第几行
    col_indices = valid_idx % W  # 每个有效像素属于第几列

    unique_rows = np.unique(row_indices)
    X_all = np.empty((len(valid_idx), B), dtype=np.float32)
    ptr = 0
    for r in unique_rows:
        mask_r = (row_indices == r)
        cols_r = col_indices[mask_r]
        X_all[ptr: ptr + mask_r.sum()] = cube_raw[r, cols_r, :].astype(np.float32)
        ptr += mask_r.sum()
    del cube_raw, mat_cube  # 立刻释放大块内存

    unique_labels = sorted(np.unique(mask_val))
    y_all = mask_val.astype(np.int32)
    label_map = {l: l for l in unique_labels}

    return X_all, y_all, wvl, label_map


def load_single_cube_lazy(mat_path,
                          cube_key="cubo",
                          mask_path=None,
                          mask_bmp_folder=None,
                          mask_key=None,
                          wvl_key=None,
                          exclude_bg_label=9,
                          max_pixels_per_cube=None,
                          seed=42):
    """
    Lazy 单 cube 读取：
    - 只在真正需要时加载这个 cube
    - 先去背景，再随机保留最多 max_pixels_per_cube 个像素
    - 返回采样后的 X, y, wvl, label_map
    """
    mat_cube = _loadmat_any(mat_path)
    cube_raw = mat_cube[cube_key]
    H, W, B = cube_raw.shape
    n_pixels = H * W

    # wavelength
    if wvl_key and wvl_key in mat_cube:
        wvl = mat_cube[wvl_key].ravel().astype(float)
    else:
        wvl = WAVELENGTHS_SWIR[:B] if B <= len(WAVELENGTHS_SWIR) else WAVELENGTHS_SWIR

    # mask
    if mask_bmp_folder is not None:
        raw_mask, _bmp_names = load_mask_folder(mask_bmp_folder)
        if raw_mask.shape != (H, W):
            raise ValueError(
                f"BMP mask spatial size {raw_mask.shape} does not match cube spatial size ({H}, {W})."
            )
        mask = raw_mask.ravel()
    else:
        mat_mask = _loadmat_any(mask_path) if mask_path is not None else mat_cube

        if mask_key is not None:
            mask = mat_mask[mask_key].astype(int).ravel()
        else:
            candidates = {
                k: v for k, v in mat_mask.items()
                if not k.startswith("__")
                   and hasattr(v, "shape")
                   and int(np.prod(v.shape)) == n_pixels
                   and k != cube_key
            }
            if not candidates:
                avail = {
                    k: v.shape for k, v in mat_mask.items()
                    if hasattr(v, "shape") and not k.startswith("__")
                }
                raise KeyError(
                    f"Cannot auto-detect mask key in '{mask_path or mat_path}'. "
                    f"cube shape=({H}, {W}, {B}), available arrays={avail}"
                )
            mask_key_found, mask_arr = next(iter(candidates.items()))
            print(f" [auto-detect] mask_key='{mask_key_found}' shape={mask_arr.shape}")
            mask = mask_arr.astype(int).ravel()

    # valid pixels
    valid_idx = np.where((mask != 0) & (mask != exclude_bg_label))[0]
    if valid_idx.size == 0:
        raise RuntimeError(f"No valid pixels left in {mat_path} after background removal.")

    # ★ 关键：在真正构造 X_all 之前先随机抽样
    rng = np.random.RandomState(seed)
    if max_pixels_per_cube is not None and len(valid_idx) > max_pixels_per_cube:
        keep = rng.choice(len(valid_idx), max_pixels_per_cube, replace=False)
        valid_idx = valid_idx[keep]

    mask_val = mask[valid_idx]

    row_indices = valid_idx // W
    col_indices = valid_idx % W

    unique_rows = np.unique(row_indices)
    X_all = np.empty((len(valid_idx), B), dtype=np.float32)

    ptr = 0
    for r in unique_rows:
        mask_r = (row_indices == r)
        cols_r = col_indices[mask_r]
        n_take = int(mask_r.sum())
        X_all[ptr: ptr + n_take] = cube_raw[r, cols_r, :].astype(np.float32)
        ptr += n_take

    del cube_raw, mat_cube

    unique_labels = sorted(np.unique(mask_val))
    # We should return the original label integers without remapping so that labels are consistent across all cubes.
    # Otherwise, class '1' in cube A might map to a different material than class '1' in cube B.
    # However, keeping the label map logic, we need to ensure the classifier sees consistent labels.
    # Since the masks from different cubes have consistent BMP names like '01_PET', their integer labels
    # are already consistent globally (e.g. 1 is always PET). We shouldn't remap them starting from 0.
    y_all = mask_val.astype(np.int32)
    label_map = {l: l for l in unique_labels}

    return X_all, y_all, wvl, label_map


def load_swir_plastics(data_dir="swir_cubes",
                       mask_dir=None,
                       mat_files=None,
                       cube_key="cubo",
                       mask_key=None,
                       wvl_key=None,
                       exclude_bg_label=9):
    """
    Load SWIR .mat cubes.
    """
    data_dir = Path(data_dir)
    mask_dir = Path(mask_dir) if mask_dir is not None else data_dir

    if mat_files is None:
        mat_files = sorted(data_dir.glob("cube*.mat"))
    else:
        mat_files = [data_dir / f for f in mat_files]

    if not mat_files:
        raise FileNotFoundError(f"No cube*.mat files found in {data_dir}")

    cubes = []
    wvl_ref = None

    for f in mat_files:
        stem = f.stem
        mask_stem = stem.replace("cube", "masks", 1)
        bmp_folder = mask_dir / mask_stem
        mat_mask_f = mask_dir / (mask_stem + ".mat")

        if bmp_folder.exists() and bmp_folder.is_dir():
            mp, bmp_dir = None, bmp_folder
        elif mat_mask_f.exists():
            mp, bmp_dir = mat_mask_f, None
        else:
            mp, bmp_dir = None, None
            print(f"  [warn] No separate mask found for {f.name}; trying inside cube.")

        print(f"  [Process] Parsing {f.name} ...")
        X, y, wvl, lmap = load_single_cube(
            f, cube_key=cube_key, mask_path=mp, mask_bmp_folder=bmp_dir,
            mask_key=mask_key, wvl_key=wvl_key, exclude_bg_label=exclude_bg_label
        )

        valid = (y != 0) & (y != exclude_bg_label)
        if not np.all(valid):
            X = X[valid]
            y = y[valid]
        n_cls = len(np.unique(y))
        if n_cls < 2:
            print(f"  [skip] {stem}: only {n_cls} class after filtering")
            continue

        cubes.append((X, y, stem))

        if wvl_ref is None:
            wvl_ref = wvl

        print(f"  {stem:<25s}  pixels={len(y):>6d}  classes={n_cls}  "
              f"bands={X.shape[1]}")

    # Update global wavelength array from file data
    if wvl_ref is not None and len(wvl_ref) > 0:
        global WAVELENGTHS, PRIOR_PEAKS_IDX, SIGMA_BANDS
        WAVELENGTHS = wvl_ref
        PRIOR_PEAKS_IDX = nm_to_idx(WAVELENGTHS, PRIOR_PEAKS_NM)
        SIGMA_BANDS = _compute_sigma_bands(WAVELENGTHS, SIGMA_NM)

    return cubes


MATERIAL_NAMES = {
    1: "PET", 2: "HDPE", 3: "LDPE",
    4: "PP",  5: "EPSF", 7: "Weathered"
}
WATER_NAMES = {0: "Clear", 1: "Turbid", 2: "Foamy"}


def load_river_csv(csv_path,
                   min_plastic_frac=0.0,
                   exclude_mix=True,
                   water_filter=None,
                   band_range=(940, 1680)):
    df = pd.read_csv(csv_path, header=None)
    df.columns = ["code", "fraction"] + list(range(350, 2501))

    df["material"] = (df["code"] // 10).astype(int)
    df["water"] = (df["code"] % 10).astype(int)

    df = df[df["fraction"] >= min_plastic_frac]
    if exclude_mix:
        df = df[df["material"] != 6]
    if water_filter is not None:
        df = df[df["water"] == water_filter]

    all_wvl = np.arange(350, 2501, dtype=float)
    band_mask = (all_wvl >= band_range[0]) & (all_wvl <= band_range[1])
    wvl = all_wvl[band_mask]
    band_cols = list(all_wvl[band_mask].astype(int))

    X = df[band_cols].values.astype(np.float32)
    y = df["material"].values.astype(np.int32)
    meta = df[["code", "material", "water", "fraction"]].reset_index(drop=True)

    print(f"[CSV] loaded {len(y)} samples | "
          f"{len(np.unique(y))} materials | "
          f"{X.shape[1]} bands ({band_range[0]}–{band_range[1]} nm)")
    for m in np.unique(y):
        print(f"  {MATERIAL_NAMES.get(int(m), int(m))}: {(y == m).sum()} samples")

    return X, y, wvl, meta


def run_csv_experiment_kfold(csv_path,
                             ratios=None,
                             n_splits=5,
                             seeds=[42, 43, 44, 45, 46],
                             min_plastic_frac=0.0,
                             water_filter=0,
                             band_range=(940, 1680),
                             sigma_nm=7.0,
                             quiet=False):
    global WAVELENGTHS, PRIOR_PEAKS_IDX, SIGMA_BANDS
    if ratios is None:
        ratios = TRAIN_RATIOS

    X_all, y_all, wvl, meta = load_river_csv(
        csv_path,
        min_plastic_frac=min_plastic_frac,
        water_filter=water_filter,
        band_range=band_range
    )

    WAVELENGTHS = wvl
    PRIOR_PEAKS_IDX = nm_to_idx(WAVELENGTHS, PRIOR_PEAKS_NM)
    SIGMA_BANDS = _compute_sigma_bands(WAVELENGTHS, sigma_nm)
    if not quiet:
        print(f"Prior peaks: {list(zip(PRIOR_PEAKS_NM, PRIOR_PEAKS_IDX))}")
        print(f"σ = {SIGMA_BANDS} bands")

    n_cls = len(np.unique(y_all))
    if n_cls < 2:
        raise ValueError("< 2 classes, check filters")

    records = []
    pc_records = []  # Store PC1/PC2 values

    for seed_idx, seed in enumerate(seeds):
        if not quiet:
            print(f"\n{'='*60}")
            print(f"Seed {seed_idx + 1}/{len(seeds)} (seed={seed})")
            print(f"{'='*60}")

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
            X_train_pool = X_all[train_idx]
            y_train_pool = y_all[train_idx]
            X_test = X_all[test_idx]
            y_test = y_all[test_idx]

            fold_seed = seed + 1000 * fold_idx
            X_test, y_test = subsample_per_class(
                X_test, y_test, max_pixels_per_class=500, seed=fold_seed
            )
            X_test = X_test.astype(np.float32, copy=False)

            X_train_pool_snv = snv(X_train_pool).astype(np.float32, copy=False)
            X_test_snv = snv(X_test).astype(np.float32, copy=False)

            total = len(y_train_pool)
            if not quiet:
                print(f"\n── Seed {seed_idx + 1}/{len(seeds)} Fold {fold_idx + 1}/{n_splits}  train={total}  test={len(y_test)}  classes={n_cls} ──")

            for ratio in ratios:
                n_tr = max(int(ratio * total), n_cls)
                if n_tr >= total:
                    Xtr = X_train_pool
                    Xtr_snv = X_train_pool_snv
                    ytr = y_train_pool
                else:
                    try:
                        Xtr, _, ytr, _ = train_test_split(
                            X_train_pool, y_train_pool,
                            train_size=n_tr, stratify=y_train_pool, random_state=fold_seed
                        )
                        Xtr_snv, _, _, _ = train_test_split(
                            X_train_pool_snv, y_train_pool,
                            train_size=n_tr, stratify=y_train_pool, random_state=fold_seed
                        )
                    except ValueError:
                        Xtr, _, ytr, _ = train_test_split(
                            X_train_pool, y_train_pool,
                            train_size=n_tr, random_state=fold_seed
                        )
                        Xtr_snv, _, _, _ = train_test_split(
                            X_train_pool_snv, y_train_pool,
                            train_size=n_tr, random_state=fold_seed
                        )
                Xtr = Xtr.astype(np.float32, copy=False)
                Xtr_snv = Xtr_snv.astype(np.float32, copy=False)

                for method_name, method_fn in METHODS.items():
                    try:
                        if method_name == "F: PLS-DA":
                            pls, classes = fit_plsda(Xtr, ytr)
                            y_pred = predict_plsda(pls, classes, X_test)
                            metrics = compute_metrics(y_test, y_pred)
                            pc1, pc2 = None, None
                        else:
                            is_snv_pre = method_name in SNV_PRE_METHODS
                            xtr_arg = Xtr_snv if is_snv_pre else Xtr
                            xte_arg = X_test_snv if is_snv_pre else X_test
                            result = method_fn(xtr_arg, xte_arg, ytr=ytr,
                                               print_variance=True, return_variance=True,
                                               pre_snv=is_snv_pre)
                            if len(result) == 4:
                                Xtr_f, Xte_f, pc1, pc2 = result
                            else:
                                Xtr_f, Xte_f = result
                                pc1, pc2 = None, None
                            skip_snv = method_name in METHODS_SKIP_POST_SNV
                            metrics = evaluate(Xtr_f, Xte_f, ytr, y_test, skip_snv=skip_snv)
                    except Exception as e:
                        metrics = {m: np.nan for m in METRIC_COLS}
                        pc1, pc2 = None, None
                        if not quiet:
                            print(f"  ⚠ {method_name} failed: {e}")

                    records.append({
                        "seed": seed,
                        "fold": fold_idx,
                        "ratio": ratio,
                        "n_train": len(ytr),
                        "method": method_name,
                        **metrics
                    })
                    
                    # Store PC1/PC2 values
                    if pc1 is not None:
                        pc_records.append({
                            "seed": seed,
                            "fold": fold_idx,
                            "ratio": ratio,
                            "method": method_name,
                            "PC1": pc1,
                            "PC2": pc2
                        })

                if not quiet:
                    row_a = next(r for r in records[-len(METHODS):]
                                 if r["method"] == "A: Global TruncSVD")
                    row_a2 = next(r for r in records[-len(METHODS):]
                                 if r["method"] == "A2: Global TruncSVD (SNV-Pre)")
                    print(f"  ratio={ratio:.1%} n_train={len(ytr):5d}  "
                          f"GlobalPCA OA={row_a['OA']:.3f} F1={row_a['macro_F1']:.3f}  "
                          f"SNV-Pre OA={row_a2['OA']:.3f} F1={row_a2['macro_F1']:.3f}" )

    return pd.DataFrame(records), pd.DataFrame(pc_records)


def run_water_ood_experiment(csv_path,
                             ratios,
                             seeds=[42, 43, 44, 45, 46],
                             n_splits=5,
                             min_plastic_frac=0.0,
                             band_range=(940, 1680),
                             sigma_nm=7.0,
                             train_water=0,
                             id_test_size=0.2,
                             test_max_per_class=500,
                             use_multi_train=False):
    global WAVELENGTHS, PRIOR_PEAKS_IDX, SIGMA_BANDS

    X_all, y_all, wvl, meta = load_river_csv(
        csv_path,
        min_plastic_frac=min_plastic_frac,
        water_filter=None,
        band_range=band_range
    )

    WAVELENGTHS = wvl
    PRIOR_PEAKS_IDX = nm_to_idx(WAVELENGTHS, PRIOR_PEAKS_NM)
    SIGMA_BANDS = _compute_sigma_bands(WAVELENGTHS, sigma_nm)

    water = meta["water"].values.astype(int)
    mask_train = water == 0
    X_clear = X_all[mask_train]
    y_clear = y_all[mask_train]

    n_cls = len(np.unique(y_clear))
    if n_cls < 2:
        raise ValueError("< 2 classes in clear-water split, check filters")

    records = []

    for seed_idx, seed in enumerate(seeds):
        print(f"\n{'='*60}")
        print(f"OOD Seed {seed_idx + 1}/{len(seeds)} (seed={seed})")
        print(f"{'='*60}")

        # Use StratifiedKFold for clear data (5-fold CV)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        # Pre-load turbid and foamy data
        X_turb_all = X_all[water == 1]
        y_turb_all = y_all[water == 1]
        X_foam_all = X_all[water == 2]
        y_foam_all = y_all[water == 2]

        # Pre-build k-fold splits for turbid/foamy when multi-background training
        # is active.  Splitting *before* the fold loop guarantees that the
        # turbid/foamy pixels used for training are never used for OOD testing.
        if use_multi_train:
            has_turb = len(y_turb_all) >= n_splits
            has_foam = len(y_foam_all) >= n_splits
            skf_turb = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + 7)
            skf_foam = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + 13)
            turb_folds = list(skf_turb.split(X_turb_all, y_turb_all)) if has_turb else None
            foam_folds = list(skf_foam.split(X_foam_all, y_foam_all)) if has_foam else None
        else:
            # Without multi-train, all turbid/foamy serve as OOD test only (no
            # overlap risk).  Pre-compute once per seed.
            X_turb = X_turb_all
            y_turb = y_turb_all
            X_foam = X_foam_all
            y_foam = y_foam_all
            X_turb, y_turb = subsample_per_class(X_turb, y_turb, max_pixels_per_class=test_max_per_class, seed=seed + 2)
            X_foam, y_foam = subsample_per_class(X_foam, y_foam, max_pixels_per_class=test_max_per_class, seed=seed + 3)
            X_turb = X_turb.astype(np.float32, copy=False)
            X_foam = X_foam.astype(np.float32, copy=False)
            X_turb_illum = ood_illum(X_turb, std=0.35, seed=seed + 11).astype(np.float32, copy=False)
            X_turb_drift = ood_sensor_drift(X_turb, sigma=0.02, seed=seed + 12).astype(np.float32, copy=False)
            X_turb_shift = ood_band_shift(X_turb, shift_bands=2).astype(np.float32, copy=False)

        # Inner loop: n-fold CV — each background contributes its own fold split
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_clear, y_clear)):
            fold_seed = seed * 1000 * fold_idx
            
            X_clear_train = X_clear[train_idx]
            y_clear_train = y_clear[train_idx]
            X_id = X_clear[test_idx]
            y_id = y_clear[test_idx]

            # ── Multi-background: fold-level train/test split for turbid & foamy ──
            if use_multi_train:
                if turb_folds is not None:
                    tr_idx, te_idx = turb_folds[fold_idx]
                    X_turb_train = X_turb_all[tr_idx];   y_turb_train = y_turb_all[tr_idx]
                    X_turb = X_turb_all[te_idx];          y_turb = y_turb_all[te_idx]
                else:
                    X_turb_train = X_turb_all;            y_turb_train = y_turb_all
                    X_turb = X_turb_all;                  y_turb = y_turb_all
                if foam_folds is not None:
                    tr_idx, te_idx = foam_folds[fold_idx]
                    X_foam_train = X_foam_all[tr_idx];   y_foam_train = y_foam_all[tr_idx]
                    X_foam = X_foam_all[te_idx];          y_foam = y_foam_all[te_idx]
                else:
                    X_foam_train = X_foam_all;            y_foam_train = y_foam_all
                    X_foam = X_foam_all;                  y_foam = y_foam_all
                X_turb, y_turb = subsample_per_class(X_turb, y_turb, max_pixels_per_class=test_max_per_class, seed=fold_seed + 2)
                X_foam, y_foam = subsample_per_class(X_foam, y_foam, max_pixels_per_class=test_max_per_class, seed=fold_seed + 3)
                X_turb = X_turb.astype(np.float32, copy=False)
                X_foam = X_foam.astype(np.float32, copy=False)
                X_turb_illum = ood_illum(X_turb, std=0.35, seed=fold_seed + 11).astype(np.float32, copy=False)
                X_turb_drift = ood_sensor_drift(X_turb, sigma=0.02, seed=fold_seed + 12).astype(np.float32, copy=False)
                X_turb_shift = ood_band_shift(X_turb, shift_bands=2).astype(np.float32, copy=False)

            print(f"\n── Seed {seed_idx + 1}/{len(seeds)} Fold {fold_idx + 1}/{n_splits}  train={len(y_clear_train)}  test={len(y_id)} ──")

            # Subsample X_id
            X_id, y_id = subsample_per_class(X_id, y_id, max_pixels_per_class=test_max_per_class, seed=seed * 100 + fold_idx)
            X_id = X_id.astype(np.float32, copy=False)

            # ID-based OOD perturbations (regenerated for each fold)
            X_id_illum = ood_illum(X_id, std=0.35, seed=seed + 21 + fold_idx).astype(np.float32, copy=False)
            X_id_drift = ood_sensor_drift(X_id, sigma=0.02, seed=seed + 22 + fold_idx).astype(np.float32, copy=False)
            X_id_shift = ood_band_shift(X_id, shift_bands=2).astype(np.float32, copy=False)

            # Build training pool
            if use_multi_train:
                X_train_pool = np.vstack([X_clear_train, X_turb_train, X_foam_train])
                y_train_pool = np.concatenate([y_clear_train, y_turb_train, y_foam_train])
            else:
                X_train_pool = X_clear_train
                y_train_pool = y_clear_train

            total = len(y_train_pool)

            X_train_pool_snv = snv(X_train_pool).astype(np.float32, copy=False)

            for ratio in ratios:
                n_tr = max(int(ratio * total), n_cls)
                if n_tr >= total:
                    Xtr = X_train_pool
                    Xtr_snv = X_train_pool_snv
                    ytr = y_train_pool
                else:
                    try:
                        Xtr, _, ytr, _ = train_test_split(
                            X_train_pool, y_train_pool,
                            train_size=n_tr,
                            stratify=y_train_pool,
                            random_state=fold_seed
                        )
                        Xtr_snv, _, _, _ = train_test_split(
                            X_train_pool_snv, y_train_pool,
                            train_size=n_tr,
                            stratify=y_train_pool,
                            random_state=fold_seed
                        )
                    except ValueError:
                        Xtr, _, ytr, _ = train_test_split(
                            X_train_pool, y_train_pool,
                            train_size=n_tr,
                            random_state=fold_seed
                        )
                        Xtr_snv, _, _, _ = train_test_split(
                            X_train_pool_snv, y_train_pool,
                            train_size=n_tr,
                            random_state=fold_seed
                        )
                Xtr = Xtr.astype(np.float32, copy=False)
                Xtr_snv = Xtr_snv.astype(np.float32, copy=False)

                for method_name, method_fn in METHODS.items():
                    row = {"seed": seed, "fold": fold_idx, "ratio": ratio, "method": method_name}
                    try:
                        test_entries = [
                            (X_id, y_id, "ID"),
                            (X_turb, y_turb, "OOD-Turbid"),
                            (X_foam, y_foam, "OOD-Foamy"),
                            (X_id_illum, y_id, "ID-IllumShift"),
                            (X_id_drift, y_id, "ID-SensorDrift"),
                            (X_id_shift, y_id, "ID-BandShift"),
                            (X_turb_illum, y_turb, "OOD-IllumShift"),
                            (X_turb_drift, y_turb, "OOD-SensorDrift"),
                            (X_turb_shift, y_turb, "OOD-BandShift"),
                        ]
                        valid = [(X, y, n) for X, y, n in test_entries if len(y) > 0]

                        if method_name == "F: PLS-DA":
                            pls, classes = fit_plsda(Xtr, ytr)
                            if valid:
                                all_Xte = np.vstack([X for X, y, _ in valid])
                                all_y_pred = predict_plsda(pls, classes, all_Xte)
                                split_sizes = [len(y) for _, y, _ in valid]
                                pred_splits = np.split(all_y_pred, np.cumsum(split_sizes)[:-1])
                                for y_pred, y_cond, name in zip(pred_splits, [y for _, y, _ in valid], [n for _, _, n in valid]):
                                    row[name] = float(accuracy_score(y_cond, y_pred))
                                    row[name + "_F1"] = float(f1_score(y_cond, y_pred, average='macro'))
                            for X, y, name in test_entries:
                                if len(y) == 0:
                                    row[name] = np.nan
                                    row[name + "_F1"] = np.nan
                        else:
                            is_snv_pre = method_name in SNV_PRE_METHODS
                            xtr_arg = Xtr_snv if is_snv_pre else Xtr
                            if valid:
                                all_Xte = np.vstack([X for X, y, _ in valid])
                                if is_snv_pre:
                                    all_Xte = snv(all_Xte).astype(np.float32, copy=False)
                                split_sizes = [len(y) for _, y, _ in valid]
                                Xtr_f, all_Xte_f = method_fn(xtr_arg, all_Xte, ytr=ytr, pre_snv=is_snv_pre)
                                skip_snv = method_name in METHODS_SKIP_POST_SNV
                                if not skip_snv:
                                    Xtr_f = snv(Xtr_f)
                                    all_Xte_f = snv(all_Xte_f)
                                sc = StandardScaler()
                                clf = make_clf()
                                clf.fit(sc.fit_transform(Xtr_f), ytr)
                                te_splits = np.split(all_Xte_f, np.cumsum(split_sizes)[:-1])
                                for Xte_f, y_cond, name in zip(te_splits, [y for _, y, _ in valid], [n for _, _, n in valid]):
                                    y_pred = clf.predict(sc.transform(Xte_f))
                                    row[name] = float(accuracy_score(y_cond, y_pred))
                                    row[name + "_F1"] = float(f1_score(y_cond, y_pred, average='macro', zero_division=0))
                            for X, y, name in test_entries:
                                if len(y) == 0:
                                    row[name] = np.nan
                                    row[name + "_F1"] = np.nan
                    except Exception:
                        row.update({
                            "ID": np.nan,
                            "OOD-Turbid": np.nan,
                            "OOD-Foamy": np.nan,
                            "ID-IllumShift": np.nan,
                            "ID-SensorDrift": np.nan,
                            "ID-BandShift": np.nan,
                            "OOD-IllumShift": np.nan,
                            "OOD-SensorDrift": np.nan,
                            "OOD-BandShift": np.nan,
                            "ID_F1": np.nan,
                            "OOD-Turbid_F1": np.nan,
                            "OOD-Foamy_F1": np.nan,
                            "ID-IllumShift_F1": np.nan,
                            "ID-SensorDrift_F1": np.nan,
                            "ID-BandShift_F1": np.nan,
                            "OOD-IllumShift_F1": np.nan,
                            "OOD-SensorDrift_F1": np.nan,
                            "OOD-BandShift_F1": np.nan,
                        })

                    records.append(row)

    cols = ["seed", "fold", "ratio", "method",
            "ID", "OOD-Turbid", "OOD-Foamy",
            "ID-IllumShift", "ID-SensorDrift", "ID-BandShift",
            "OOD-IllumShift", "OOD-SensorDrift", "OOD-BandShift",
            "ID_F1", "OOD-Turbid_F1", "OOD-Foamy_F1",
            "ID-IllumShift_F1", "ID-SensorDrift_F1", "ID-BandShift_F1",
            "OOD-IllumShift_F1", "OOD-SensorDrift_F1", "OOD-BandShift_F1"]
    return pd.DataFrame(records)[cols]


def plot_ood_heatmap(df, ratio, out_dir="."):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)
    conds = ["ID", "OOD-Turbid", "OOD-Foamy",
             "ID-IllumShift", "ID-SensorDrift", "ID-BandShift",
             "OOD-IllumShift", "OOD-SensorDrift", "OOD-BandShift"]
    cond_mean_cols = [f'{c}_mean' for c in conds]
    sub = df[df["ratio"] == ratio].copy()
    if sub.empty:
        return
    sub = sub.set_index("method")[cond_mean_cols]
    methods = list(sub.index)
    M = (sub.values.astype(float) * 100.0)

    fig, ax = plt.subplots(figsize=(12, max(3.5, 0.35 * len(methods))))
    im = ax.imshow(M, aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)
    ax.set_yticks(np.arange(len(methods)))
    ax.set_yticklabels(methods, fontsize=9)
    ax.set_xticks(np.arange(len(conds)))
    ax.set_xticklabels(conds, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"OOD (water) — OA (%)  ratio={ratio:.0%}", fontsize=12)
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    fig.tight_layout()
    out = out_dir / f"ood_water_heatmap_ratio_{int(round(ratio*100)):02d}.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Saved: {out}")


def make_synthetic_data(n_samples=5000, n_bands=60,
                        n_classes=5, snr_db=12, seed=42):
    """
    ⚠  SANITY CHECK ONLY — circular reasoning risk (see module docstring).
    Class differences encoded at PRIOR_PEAKS_IDX; strong illumination fills
    the first PCA components.
    """
    rng = np.random.RandomState(seed)
    wvl = WAVELENGTHS[:n_bands]
    peak_idxs = nm_to_idx(wvl, PRIOR_PEAKS_NM)
    sigma = _compute_sigma_bands(wvl, SIGMA_NM)
    sizes = rng.multinomial(n_samples, [1 / n_classes] * n_classes)
    depths = rng.uniform(0.05, 0.40, size=(n_classes, len(peak_idxs)))

    X_list, y_list = [], []
    for cls in range(n_classes):
        n = sizes[cls]
        illum = 2.0 * rng.randn(n, 1)
        slope = 0.8 * rng.randn(n, 1) * np.linspace(-1, 1, n_bands)
        spec = np.ones((n, n_bands)) * (1 + illum * 0.3) + slope
        for i, pidx in enumerate(peak_idxs):
            g = depths[cls, i] * np.exp(
                -0.5 * ((np.arange(n_bands) - pidx) / sigma) ** 2)
            spec -= g
        sig_pwr = np.mean(spec ** 2)
        spec += np.sqrt(sig_pwr / 10 ** (snr_db / 10)) * rng.randn(n, n_bands)
        X_list.append(spec.astype(np.float32))
        y_list.append(np.full(n, cls, dtype=int))

    return np.vstack(X_list), np.concatenate(y_list)


# ─── 2. Preprocessing ─────────────────────────────────────────────────────────


def complement_bands(n_bands, peak_idxs, sigma_mult=3, sigma=None):
    if sigma is None:
        sigma = SIGMA_BANDS
    used = np.zeros(n_bands, dtype=bool)
    for c in peak_idxs:
        lo = max(0, int(c - sigma_mult * sigma))
        hi = min(n_bands, int(c + sigma_mult * sigma) + 1)
        used[lo:hi] = True
    return np.where(~used)[0]


def prior_band_mask(n_bands, peak_idxs, sigma_mult=3, sigma=None):
    comp = complement_bands(n_bands, peak_idxs, sigma_mult, sigma)
    mask = np.ones(n_bands, dtype=bool)
    mask[comp] = False
    return mask


def _comp_pca_fit(Xtr, n_comp, center=False, print_variance=False, method_name=""):
    """Fit TruncatedSVD on training data.
    
    Parameters
    ----------
    Xtr : array (n_samples, n_features)
    n_comp : int — number of components
    center : bool — if True, subtract mean before fitting (centered control)
    print_variance : bool — if True, print PC1/PC2 variance ratio
    method_name : str — method name for printing
    
    Returns
    -------
    svd_obj : fitted TruncatedSVD object
    mu : mean vector or None (only when center=True)
    pc1_ratio : float or None — PC1 variance ratio (0-100)
    pc2_ratio : float or None — PC2 variance ratio (0-100)
    """
    n, p = Xtr.shape
    n_comp = max(1, min(n_comp, min(n, p) - 1 if min(n, p) > 1 else 1, p))
    mu = None
    X_fit = Xtr
    if center:
        mu = Xtr.mean(axis=0)
        X_fit = Xtr - mu
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    svd.fit(X_fit)
    sv_sq = svd.singular_values_ ** 2         # 严格降序
    total_energy = np.sum(X_fit ** 2)          # Frobenius 范数平方
    
    pc1_ratio = sv_sq[0] / total_energy * 100
    pc2_ratio = (sv_sq[1] / total_energy * 100) if len(sv_sq) > 1 else None
    
    return svd, mu, pc1_ratio, pc2_ratio


def _comp_pca_transform(Xte, svd_obj, mu=None):
    """Transform data using fitted TruncatedSVD. Subtract mu if provided."""
    X = Xte - mu if mu is not None else Xte
    return svd_obj.transform(X)



# ─── 3. Dimensionality Reduction Methods ──────────────────────────────────────

def method_global_pca(Xtr, Xte, n_dim=N_DIM, **kw):
    """A: Global TruncatedSVD, no centering."""
    print_var = kw.get('print_variance', False)
    return_var = kw.get('return_variance', False)
    svd, mu, pc1, pc2 = _comp_pca_fit(Xtr, n_dim, center=False, print_variance=print_var, method_name="A: Global PCA")
    Xtr_f, Xte_f = _comp_pca_transform(Xtr, svd, mu), _comp_pca_transform(Xte, svd, mu)
    return (Xtr_f, Xte_f, pc1, pc2) if return_var else (Xtr_f, Xte_f)

def method_global_pca_snv_pre(Xtr, Xte, n_dim=N_DIM, **kw):
    """A2: SNV → Global TruncatedSVD, no centering."""
    print_var = kw.get('print_variance', False)
    return_var = kw.get('return_variance', False)
    if not kw.get('pre_snv', False):
        Xtr = snv(Xtr)
        Xte = snv(Xte)
    svd, mu, pc1, pc2 = _comp_pca_fit(Xtr, n_dim, center=False, print_variance=print_var, method_name="A2: SNV-Pre")
    Xtr_f, Xte_f = _comp_pca_transform(Xtr, svd, mu), _comp_pca_transform(Xte, svd, mu)
    return (Xtr_f, Xte_f, pc1, pc2) if return_var else (Xtr_f, Xte_f)

def method_global_pca_centered(Xtr, Xte, n_dim=N_DIM, **kw):
    """A*: Global TruncatedSVD, with centering (control for A)."""
    print_var = kw.get('print_variance', False)
    return_var = kw.get('return_variance', False)
    svd, mu, pc1, pc2 = _comp_pca_fit(Xtr, n_dim, center=True, print_variance=print_var, method_name="A*: Centered")
    Xtr_f, Xte_f = _comp_pca_transform(Xtr, svd, mu), _comp_pca_transform(Xte, svd, mu)
    return (Xtr_f, Xte_f, pc1, pc2) if return_var else (Xtr_f, Xte_f)

def method_global_pca_snv_pre_centered(Xtr, Xte, n_dim=N_DIM, **kw):
    """A2*: SNV → Global TruncatedSVD, with centering (control for A2)."""
    print_var = kw.get('print_variance', False)
    return_var = kw.get('return_variance', False)
    if not kw.get('pre_snv', False):
        Xtr = snv(Xtr)
        Xte = snv(Xte)
    svd, mu, pc1, pc2 = _comp_pca_fit(Xtr, n_dim, center=True, print_variance=print_var, method_name="A2*: SNV-Pre Centered")
    Xtr_f, Xte_f = _comp_pca_transform(Xtr, svd, mu), _comp_pca_transform(Xte, svd, mu)
    return (Xtr_f, Xte_f, pc1, pc2) if return_var else (Xtr_f, Xte_f)

def method_global_pca_msc_pre(Xtr, Xte, n_dim=N_DIM, **kw):
    """A3: MSC → Global TruncatedSVD, no centering.
    MSC reference fitted on training set only to prevent information leakage.
    """
    print_var = kw.get('print_variance', False)
    return_var = kw.get('return_variance', False)
    msc = MSC()
    Xtr_msc = msc.fit_transform(Xtr)
    Xte_msc = msc.transform(Xte)
    svd, mu, pc1, pc2 = _comp_pca_fit(Xtr_msc, n_dim, center=False, print_variance=print_var, method_name="A3: MSC-Pre")
    Xtr_f, Xte_f = _comp_pca_transform(Xtr_msc, svd, mu), _comp_pca_transform(Xte_msc, svd, mu)
    return (Xtr_f, Xte_f, pc1, pc2) if return_var else (Xtr_f, Xte_f)

def method_global_pca_msc_pre_centered(Xtr, Xte, n_dim=N_DIM, **kw):
    """A3*: MSC → Global TruncatedSVD, with centering.
    MSC reference fitted on training set only to prevent information leakage.
    """
    print_var = kw.get('print_variance', False)
    return_var = kw.get('return_variance', False)
    msc = MSC()
    Xtr_msc = msc.fit_transform(Xtr)
    Xte_msc = msc.transform(Xte)
    svd, mu, pc1, pc2 = _comp_pca_fit(Xtr_msc, n_dim, center=True, print_variance=print_var, method_name="A3*: MSC-Pre Centered")
    Xtr_f, Xte_f = _comp_pca_transform(Xtr_msc, svd, mu), _comp_pca_transform(Xte_msc, svd, mu)
    return (Xtr_f, Xte_f, pc1, pc2) if return_var else (Xtr_f, Xte_f)

def method_global_truncsvd_drop1(Xtr, Xte, n_dim=N_DIM, **kw):
    svd, mu, _, _ = _comp_pca_fit(Xtr, n_dim, center=False)
    Ztr = _comp_pca_transform(Xtr, svd, mu)
    Zte = _comp_pca_transform(Xte, svd, mu)
    return Ztr[:, 1:], Zte[:, 1:]

def method_global_truncsvd_l2(Xtr, Xte, n_dim=N_DIM, **kw):
    svd, mu, _, _ = _comp_pca_fit(Xtr, n_dim, center=False)
    Ztr = _comp_pca_transform(Xtr, svd, mu)
    Zte = _comp_pca_transform(Xte, svd, mu)
    return normalize(Ztr, norm='l2', axis=1), normalize(Zte, norm='l2', axis=1)

def method_global_truncsvd_drop1_scale(Xtr, Xte, n_dim=N_DIM, **kw):
    svd, mu, _, _ = _comp_pca_fit(Xtr, n_dim, center=False)
    Ztr = _comp_pca_transform(Xtr, svd, mu)
    Zte = _comp_pca_transform(Xte, svd, mu)
    return Ztr[:, 1:], Zte[:, 1:]

def method_raw_peak_only(Xtr, Xte, peak_idxs=None, sigma=None,
                         sigma_mult=3, **kw):
    """B2 — Raw bands inside prior windows; no weighting, no complement."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    sigma = sigma or SIGMA_BANDS
    mask = prior_band_mask(Xtr.shape[1], peak_idxs,
                           sigma_mult=sigma_mult, sigma=sigma)
    return Xtr[:, mask], Xte[:, mask]

def method_peak_point(Xtr, Xte, peak_idxs=None, **kw):
    # B3: 仅取峰顶单点，无窗口，无加权
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    return Xtr[:, peak_idxs], Xte[:, peak_idxs]

def method_point_comp(Xtr, Xte, peak_idxs=None, ndim=N_DIM, **kw):
    """C3: Peak-point prior + complement TruncatedSVD."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    n_prior = len(peak_idxs)
    n_comp = ndim - n_prior
    comp_idx = complement_bands(Xtr.shape[1], peak_idxs)
    pr_tr = Xtr[:, peak_idxs]
    pr_te = Xte[:, peak_idxs]
    svd, mu, _, _ = _comp_pca_fit(Xtr[:, comp_idx], n_comp, center=False)
    return (np.hstack([pr_tr, _comp_pca_transform(Xtr[:, comp_idx], svd, mu)]),
            np.hstack([pr_te, _comp_pca_transform(Xte[:, comp_idx], svd, mu)]))

def method_raw_comp(Xtr, Xte, peak_idxs=None, sigma=None,
                    sigma_mult=3, n_dim=N_DIM, **kw):
    """C2 — Raw prior bands + complement TruncatedSVD."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    sigma = sigma or SIGMA_BANDS
    prior_mask = prior_band_mask(Xtr.shape[1], peak_idxs,
                                 sigma_mult=sigma_mult, sigma=sigma)
    comp_idx = complement_bands(Xtr.shape[1], peak_idxs,
                                sigma_mult=sigma_mult, sigma=sigma)
    raw_tr = Xtr[:, prior_mask]
    raw_te = Xte[:, prior_mask]
    n_comp = max(0, n_dim - raw_tr.shape[1])
    if n_comp == 0:
        return raw_tr, raw_te
    svd, mu, _, _ = _comp_pca_fit(Xtr[:, comp_idx], n_comp, center=False)
    return (np.hstack([raw_tr, _comp_pca_transform(Xtr[:, comp_idx], svd, mu)]),
            np.hstack([raw_te, _comp_pca_transform(Xte[:, comp_idx], svd, mu)]))

def method_rect_only(Xtr, Xte, peak_idxs=None, sigma=None, sigma_mult=3,
                     half_w=None, **kw):
    """D2 — Rectangular-window mean only (no complement PCA)."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    sigma = sigma or SIGMA_BANDS
    if half_w is None:
        half_w = int(round(sigma_mult * sigma))
    n_bands = Xtr.shape[1]
    pr_tr, pr_te = [], []
    for c in peak_idxs:
        lo = max(0, c - half_w)
        hi = min(n_bands, c + half_w + 1)
        pr_tr.append(Xtr[:, lo:hi].mean(axis=1, keepdims=True))
        pr_te.append(Xte[:, lo:hi].mean(axis=1, keepdims=True))
    return np.hstack(pr_tr), np.hstack(pr_te)


def method_rect_comp(Xtr, Xte, peak_idxs=None, sigma=None, sigma_mult=3,
                     half_w=None, n_dim=N_DIM, **kw):
    """D — Rectangular-window mean + complement TruncatedSVD."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    sigma = sigma or SIGMA_BANDS
    if half_w is None:
        half_w = int(round(sigma_mult * sigma))
    n_bands = Xtr.shape[1]
    comp_idx = complement_bands(n_bands, peak_idxs, sigma_mult=sigma_mult, sigma=sigma)
    pr_tr, pr_te = [], []
    for c in peak_idxs:
        lo = max(0, c - half_w);
        hi = min(n_bands, c + half_w + 1)
        pr_tr.append(Xtr[:, lo:hi].mean(axis=1, keepdims=True))
        pr_te.append(Xte[:, lo:hi].mean(axis=1, keepdims=True))
    pr_tr = np.hstack(pr_tr);
    pr_te = np.hstack(pr_te)
    n_comp = n_dim - len(peak_idxs)
    svd, mu, _, _ = _comp_pca_fit(Xtr[:, comp_idx], n_comp, center=False)
    return (np.hstack([pr_tr, _comp_pca_transform(Xtr[:, comp_idx], svd, mu)]),
            np.hstack([pr_te, _comp_pca_transform(Xte[:, comp_idx], svd, mu)]))

def method_comp_only(Xtr, Xte, peak_idxs=None, n_dim=N_DIM, **kw):
    """E2 — Complement TruncatedSVD only (no rectangular features)."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    comp_idx = complement_bands(Xtr.shape[1], peak_idxs)
    svd, mu, _, _ = _comp_pca_fit(Xtr[:, comp_idx], n_dim, center=False)
    return (_comp_pca_transform(Xtr[:, comp_idx], svd, mu),
            _comp_pca_transform(Xte[:, comp_idx], svd, mu))

def method_local_comp(Xtr, Xte, peak_idxs=None, sigma=None, sigma_mult=3,
                      half_w=None, n_local=1, n_dim=N_DIM, **kw):
    """E — Local (interval) TruncatedSVD at each peak + complement TruncatedSVD."""
    peak_idxs = peak_idxs or PRIOR_PEAKS_IDX
    sigma = sigma or SIGMA_BANDS
    if half_w is None:
        half_w = int(round(sigma_mult * sigma))
    n_bands = Xtr.shape[1]
    n_local_tot = len(peak_idxs) * n_local
    prior_set = set()
    loc_tr, loc_te = [], []
    for c in peak_idxs:
        lo = max(0, c - half_w);
        hi = min(n_bands, c + half_w + 1)
        bw = list(range(lo, hi));
        prior_set.update(bw)
        k = min(n_local, len(bw))
        svd_l, mu_l, _, _ = _comp_pca_fit(Xtr[:, bw], k, center=False)
        loc_tr.append(_comp_pca_transform(Xtr[:, bw], svd_l, mu_l))
        loc_te.append(_comp_pca_transform(Xte[:, bw], svd_l, mu_l))
    loc_tr = np.hstack(loc_tr);
    loc_te = np.hstack(loc_te)
    comp_idx = np.array(sorted(set(range(n_bands)) - prior_set))
    n_comp = n_dim - n_local_tot
    svd, mu, _, _ = _comp_pca_fit(Xtr[:, comp_idx], n_comp, center=False)
    return (np.hstack([loc_tr, _comp_pca_transform(Xtr[:, comp_idx], svd, mu)]),
            np.hstack([loc_te, _comp_pca_transform(Xte[:, comp_idx], svd, mu)]))


def method_plsda(Xtr, Xte, n_components=N_DIM, **kw):
    y = kw.get("ytr", None)
    if y is None:
        raise ValueError("PLS-DA requires ytr passed as keyword argument.")
    y = np.asarray(y)
    y = y.astype(int)
    classes = np.unique(y)
    y_map = {c: i for i, c in enumerate(classes)}
    y_idx = np.vectorize(y_map.get)(y)
    Y = np.eye(len(classes), dtype=np.float32)[y_idx]
    k = int(min(n_components, max(1, len(classes) - 1), Xtr.shape[1], Xtr.shape[0] - 1))
    pls = PLSRegression(n_components=k)
    pls.fit(Xtr, Y)
    Xtr_f = pls.x_scores_.astype(np.float32, copy=False)
    Xte_f = pls.transform(Xte).astype(np.float32, copy=False)
    return Xtr_f, Xte_f


def fit_plsda(Xtr, ytr, n_components=N_DIM):
    y = np.asarray(ytr).astype(int)
    classes = np.unique(y)
    y_map = {c: i for i, c in enumerate(classes)}
    y_idx = np.vectorize(y_map.get)(y)
    Y = np.eye(len(classes), dtype=np.float32)[y_idx]

    k = int(min(n_components, max(1, len(classes) - 1), Xtr.shape[1], Xtr.shape[0] - 1))
    if k < 1:
        raise ValueError("PLS-DA requires at least 2 training samples and >=1 component.")

    pls = PLSRegression(n_components=k)
    pls.fit(snv(Xtr), Y)
    return pls, classes


def predict_plsda(pls, classes, Xte):
    Yhat = pls.predict(snv(Xte))
    idx = np.argmax(Yhat, axis=1)
    return classes[idx]


METHODS = {
    "A: Global TruncSVD": method_global_pca,
    "A*: Global TruncSVD (Centered)": method_global_pca_centered,
    "A2: Global TruncSVD (SNV-Pre)": method_global_pca_snv_pre,
    "A2*: Global TruncSVD (SNV-Pre, Centered)": method_global_pca_snv_pre_centered,
    "A3: Global TruncSVD (MSC-Pre)": method_global_pca_msc_pre,
    "A3*: Global TruncSVD (MSC-Pre, Centered)": method_global_pca_msc_pre_centered,
    "A_drop1: Drop PC1 (PC2–PC20)": method_global_truncsvd_drop1,
    "A_L2: L2 row norm (no SNV)": method_global_truncsvd_l2,
    "A_drop1_scale: Drop PC1 + Scale (no SNV)": method_global_truncsvd_drop1_scale,
    "F: PLS-DA": method_plsda,
}

METHODS_SKIP_POST_SNV = {"A2: Global TruncSVD (SNV-Pre)", "A2*: Global TruncSVD (SNV-Pre, Centered)",
                          "A3: Global TruncSVD (MSC-Pre)", "A3*: Global TruncSVD (MSC-Pre, Centered)",
                          "A_L2: L2 row norm (no SNV)", "A_drop1_scale: Drop PC1 + Scale (no SNV)",
                          "F: PLS-DA"}

SNV_PRE_METHODS = {"A2: Global TruncSVD (SNV-Pre)", "A2*: Global TruncSVD (SNV-Pre, Centered)"}


# ─── 4. Classifier ────────────────────────────────────────────────────────────

def make_clf():
    return LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        C=1.0,
        solver="lbfgs",
        multi_class="multinomial",
    )
    '''
    return SVC(kernel='linear', C=1,
              
               class_weight='balanced', random_state=42)
    '''

# ─── 5b. Training Subsampler ──────────────────────────────────────────────────

def subsample_per_class(X, y, max_pixels_per_class=None, seed=42):
    """
    Randomly subsample pixels to at most max_pixels_per_class per class.
    """
    if max_pixels_per_class is None:
        return X, y
    rng = np.random.RandomState(seed)
    keep = []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        if len(idx) > max_pixels_per_class:
            idx = rng.choice(idx, max_pixels_per_class, replace=False)
        keep.append(idx)
    keep = np.concatenate(keep)
    return X[keep], y[keep]


# ─── 5. Metrics ───────────────────────────────────────────────────────────────

METRIC_COLS = ["OA", "AA", "macro_F1", "kappa"]


def compute_metrics(y_true, y_pred):
    """
    Returns dict with four standard HSI metrics.
      OA       = Overall Accuracy      (pixel-weighted)
      AA       = Average Accuracy      = macro-recall (balanced)
      macro_F1 = macro-averaged F1     (precision + recall, equal class weight)
      kappa    = Cohen's kappa         (above-chance performance)
    """
    return {
        "OA": float(accuracy_score(y_true, y_pred)),
        "AA": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_F1": float(f1_score(y_true, y_pred,
                                   average="macro", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }


def snv(X, eps=1e-8):
    X = X.astype(np.float32, copy=False)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return (X - mu) / (sd + eps)


class MSC:
    def __init__(self):
        self.ref_ = None

    def fit(self, X):
        self.ref_ = X.mean(axis=0)
        return self

    def transform(self, X):
        ref = self.ref_
        ref_mean = ref.mean()
        ref_centered = ref - ref_mean
        ref_var = np.mean(ref_centered ** 2)
        X_mean = X.mean(axis=1, keepdims=True)
        cov = np.mean((X - X_mean) * ref_centered, axis=1)
        b = cov / ref_var
        a = X_mean.flatten() - b * ref_mean
        b_safe = np.where(np.abs(b) > 1e-6, b, 1.0)
        X_corr = (X - a[:, None]) / b_safe[:, None]
        return X_corr

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def evaluate(Xtr_feat, Xte_feat, ytr, yte, skip_snv=False):
    if not skip_snv:
        Xtr_feat = snv(Xtr_feat)
        Xte_feat = snv(Xte_feat)
    sc = StandardScaler()
    clf = make_clf()
    clf.fit(sc.fit_transform(Xtr_feat), ytr)
    y_pred = clf.predict(sc.transform(Xte_feat))
    return compute_metrics(yte, y_pred)


def per_class_report(Xtr_feat, Xte_feat, ytr, yte,
                     class_names=None):
    """
    Return per-class precision / recall / F1 as a DataFrame.
    Use at a fixed ratio (e.g. 5 %) to show which polymer benefits most.
    """
    Xtr_feat = snv(Xtr_feat)
    Xte_feat = snv(Xte_feat)
    sc = StandardScaler()
    clf = make_clf()
    clf.fit(sc.fit_transform(Xtr_feat), ytr)
    y_pred = clf.predict(sc.transform(Xte_feat))
    report = classification_report(yte, y_pred,
                                   target_names=class_names,
                                   output_dict=True,
                                   zero_division=0)
    df = pd.DataFrame(report).T
    return df


# ─── 6. LOCO-CV ───────────────────────────────────────────────────────────────

TRAIN_RATIOS = [0.005, 0.01, 0.05, 0.10]


def run_loco_cv(cubes,
                ratios=TRAIN_RATIOS,
                ood_cube_idx=None,
                seed=42,
                train_max_pixels_per_class=None):
    """
    Leave-One-Cube-Out Cross-Validation.
    (Memory-Optimized Version: Samples indices first to avoid large np.vstack)
    """
    if ood_cube_idx is not None:
        loco_cubes = [c for i, c in enumerate(cubes) if i != ood_cube_idx]
        print(f"OOD cube (held out from LOCO): {cubes[ood_cube_idx][2]}")
    else:
        loco_cubes = cubes

    records = []
    n_folds = len(loco_cubes)

    for fold_idx, (X_test, y_test, test_name) in enumerate(loco_cubes):
        print(f"\n── Fold {fold_idx + 1}/{n_folds}  test={test_name} "
              f"({len(y_test)} pixels, {len(np.unique(y_test))} classes) ──")

        train_pool = [(X, y) for i, (X, y, _) in enumerate(loco_cubes) if i != fold_idx]

        # 【内存优化核心】：只拼接极小的 y 标签数组，不拼接庞大的 X 数据
        y_pool = np.concatenate([c[1] for c in train_pool])
        n_cls = len(np.unique(y_pool))
        total_pixels = len(y_pool)

        for ratio in ratios:
            n_tr = max(int(ratio * total_pixels), n_cls)

            # 1. 在全局索引层面做 train_test_split，获取训练集的候选索引
            global_indices = np.arange(total_pixels)
            try:
                train_idx_candidates, _, ytr_candidates, _ = train_test_split(
                    global_indices, y_pool, train_size=n_tr,
                    stratify=y_pool, random_state=seed)
            except ValueError:
                train_idx_candidates, _, ytr_candidates, _ = train_test_split(
                    global_indices, y_pool, train_size=n_tr, random_state=seed)

            # 2. 对索引进行每类最大像素数的限制（等效于 subsample_per_class）
            rng = np.random.RandomState(seed)
            keep_idx_list = []
            for cls in np.unique(ytr_candidates):
                cls_idx = train_idx_candidates[ytr_candidates == cls]
                if train_max_pixels_per_class and len(cls_idx) > train_max_pixels_per_class:
                    cls_idx = rng.choice(cls_idx, train_max_pixels_per_class, replace=False)
                keep_idx_list.append(cls_idx)

            final_train_idx = np.concatenate(keep_idx_list)
            np.random.shuffle(final_train_idx)  # 打乱顺序

            # 3. 带着计算好的最终索引，去各个 Cube 中按需提取真实的 X 数据
            Xtr_list = []
            ytr_list = []
            offset = 0
            for X_c, y_c in train_pool:
                n_c = len(y_c)
                # 筛选出属于当前 Cube 的索引并转换为相对索引
                mask = (final_train_idx >= offset) & (final_train_idx < offset + n_c)
                local_idx = final_train_idx[mask] - offset

                if len(local_idx) > 0:
                    Xtr_list.append(X_c[local_idx])
                    ytr_list.append(y_c[local_idx])
                offset += n_c

            # 由于已经进行了严格的采样，这里拼合的只剩几千个样本，内存占用微乎其微
            Xtr = np.vstack(Xtr_list)
            ytr = np.concatenate(ytr_list)

            for method_name, method_fn in METHODS.items():
                try:
                    if method_name == "F: PLS-DA":
                        pls, classes = fit_plsda(Xtr, ytr)
                        y_pred = predict_plsda(pls, classes, X_test)
                        metrics = compute_metrics(y_test, y_pred)
                    else:
                        is_snv_pre = method_name in SNV_PRE_METHODS
                        xtr_arg = snv(Xtr).astype(np.float32, copy=False) if is_snv_pre else Xtr
                        xte_arg = snv(X_test).astype(np.float32, copy=False) if is_snv_pre else X_test
                        Xtr_f, Xte_f = method_fn(xtr_arg, xte_arg, ytr=ytr, pre_snv=is_snv_pre)
                        skip_snv = method_name in METHODS_SKIP_POST_SNV
                        metrics = evaluate(Xtr_f, Xte_f, ytr, y_test, skip_snv=skip_snv)
                except Exception as e:
                    metrics = {m: np.nan for m in METRIC_COLS}
                    print(f"    ⚠  {method_name} failed: {e}")

                records.append({
                    "fold": fold_idx,
                    "cube_test": test_name,
                    "ratio": ratio,
                    "n_train": len(ytr),
                    "method": method_name,
                    **metrics,
                })

            # Print progress
            try:
                row_a = next(r for r in records[-len(METHODS):] if r["method"] == "A: Global TruncSVD")
                row_c = next(r for r in records[-len(METHODS):] if r["method"] == "C2: Raw+Comp")
                print(f"  ratio={ratio:.1%} n_train={len(ytr):5d}  "
                      f"GlobalPCA OA={row_a['OA']:.3f}  "
                      f"Raw+Comp OA={row_c['OA']:.3f}")
            except StopIteration:
                pass

    return pd.DataFrame(records)


def run_intra_cube_cv(cube_infos,
                      ratios=TRAIN_RATIOS,
                      seed=42,
                      train_max_pixels_per_class=None,
                      test_max_pixels_per_class=500,
                      max_cubes=3,
                      max_pixels_per_cube=100000,
                      cube_key="cubo",
                      mask_key=None,
                      wvl_key=None,
                      exclude_bg_label=9):
    """
    Intra-cube Evaluation:
    - Treat each cube as a separate dataset
    - Split pixels within the same cube into Train and Test
    - Evaluate algorithms purely on the individual cube to avoid cross-cube domain shift
    """
    records = []

    if max_cubes is None:
        max_cubes = len(cube_infos)
    else:
        max_cubes = min(int(max_cubes), len(cube_infos))
    used = 0

    for info in cube_infos:
        if used >= max_cubes:
            break
        print(f"\n── Cube {used + 1}/{max_cubes}  name={info['name']} ──")

        X_all, y_all, wvl, _ = load_single_cube_lazy(
            info["mat_path"],
            cube_key=cube_key,
            mask_path=info["mask_path"],
            mask_bmp_folder=info["mask_bmp_folder"],
            mask_key=mask_key,
            wvl_key=wvl_key,
            exclude_bg_label=exclude_bg_label,
            max_pixels_per_cube=max_pixels_per_cube,
            seed=seed + used
        )

        n_classes = len(np.unique(y_all))
        print(f"   Total pixels={len(y_all)} classes={n_classes}")
        if n_classes < 2:
            print("  [skip] <2 classes after filtering; skipping this cube")
            del X_all, y_all
            continue
        used += 1

        # Update global wavelength settings
        global WAVELENGTHS, PRIOR_PEAKS_IDX, SIGMA_BANDS
        if wvl is not None and len(wvl) > 0:
            WAVELENGTHS = wvl
            PRIOR_PEAKS_IDX = nm_to_idx(WAVELENGTHS, PRIOR_PEAKS_NM)
            SIGMA_BANDS = _compute_sigma_bands(WAVELENGTHS, SIGMA_NM)

        # Base 80/20 train/test split for this cube
        X_train_pool, X_test, y_train_pool, y_test = train_test_split(
            X_all, y_all, test_size=0.2, stratify=y_all, random_state=seed
        )

        # ── Strictly balance the test set to prevent Majority Class Collapse OA cheating ──
        X_test, y_test = subsample_per_class(
            X_test, y_test,
            max_pixels_per_class=test_max_pixels_per_class,
            seed=seed
        )

        # Subsample testing data if too large to prevent OOM (should be covered by subsample_per_class, but just in case)
        max_test_pixels = 20000
        if len(y_test) > max_test_pixels:
            print(f"  [warn] Subsampling test set to {max_test_pixels} to avoid OOM")
            X_test, _, y_test, _ = train_test_split(
                X_test, y_test, train_size=max_test_pixels, stratify=y_test, random_state=seed
            )

        X_test = X_test.astype(np.float32, copy=False)
        total_train_pixels = len(y_train_pool)
        n_cls = len(np.unique(y_train_pool))

        for ratio in ratios:
            n_tr = max(int(ratio * total_train_pixels), n_cls)

            try:
                Xtr_candidates, _, ytr_candidates, _ = train_test_split(
                    X_train_pool, y_train_pool,
                    train_size=n_tr,
                    stratify=y_train_pool,
                    random_state=seed
                )
            except ValueError:
                Xtr_candidates, _, ytr_candidates, _ = train_test_split(
                    X_train_pool, y_train_pool,
                    train_size=n_tr,
                    random_state=seed
                )

            # Apply per-class max pixel limit
            rng = np.random.RandomState(seed)
            Xtr_list, ytr_list = [], []
            for cls in np.unique(ytr_candidates):
                cls_mask = (ytr_candidates == cls)
                X_cls = Xtr_candidates[cls_mask]
                y_cls = ytr_candidates[cls_mask]

                if train_max_pixels_per_class is not None and len(y_cls) > train_max_pixels_per_class:
                    keep = rng.choice(len(y_cls), train_max_pixels_per_class, replace=False)
                    X_cls = X_cls[keep]
                    y_cls = y_cls[keep]

                Xtr_list.append(X_cls.astype(np.float32, copy=False))
                ytr_list.append(y_cls)

            Xtr = np.vstack(Xtr_list)
            ytr = np.concatenate(ytr_list)

            for method_name, method_fn in METHODS.items():
                try:
                    if method_name == "F: PLS-DA":
                        pls, classes = fit_plsda(Xtr, ytr)
                        y_pred = predict_plsda(pls, classes, X_test)
                        metrics = compute_metrics(y_test, y_pred)
                    else:
                        is_snv_pre = method_name in SNV_PRE_METHODS
                        xtr_arg = snv(Xtr).astype(np.float32, copy=False) if is_snv_pre else Xtr
                        xte_arg = snv(X_test).astype(np.float32, copy=False) if is_snv_pre else X_test
                        Xtr_f, Xte_f = method_fn(xtr_arg, xte_arg, ytr=ytr, pre_snv=is_snv_pre)
                        skip_snv = method_name in METHODS_SKIP_POST_SNV
                        metrics = evaluate(Xtr_f, Xte_f, ytr, y_test, skip_snv=skip_snv)
                except Exception as e:
                    metrics = {m: np.nan for m in METRIC_COLS}
                    print(f"    ⚠ {method_name} failed: {e}")

                records.append({
                    "fold": used - 1,
                    "cube_test": info["name"],
                    "ratio": ratio,
                    "n_train": len(ytr),
                    "method": method_name,
                    **metrics,
                })

            try:
                recent = records[-len(METHODS):]
                row_a = next(r for r in recent if r["method"] == "A: Global TruncSVD")
                row_c = next(r for r in recent if r["method"] == "C2: Raw+Comp")
                print(f"  ratio={ratio:.1%} n_train={len(ytr):5d}  "
                      f"GlobalPCA OA={row_a['OA']:.3f}  "
                      f"Raw+Comp OA={row_c['OA']:.3f}")
            except StopIteration:
                pass

            del Xtr, ytr

        del X_all, y_all, X_train_pool, y_train_pool, X_test, y_test

    return pd.DataFrame(records)


def summarize_loco(df_loco):
    """
    Aggregate results over all seeds and folds: mean ± std for each (ratio, method).

    Returns
    -------
    summary : DataFrame  index=(ratio, method)  cols=(OA_mean, OA_std, …)
    """
    group_cols = ["ratio", "method"]
    if "seed" in df_loco.columns:
        group_cols = ["ratio", "method"]
    
    agg = (df_loco.groupby(group_cols)[METRIC_COLS]
           .agg(["mean", "std"])
           .round(4))
    agg.columns = [f"{m}_{s}" for m, s in agg.columns]
    return agg.reset_index()


def build_report_table(summary):
    """
    Build a publication-ready table:
      rows = methods, cols = metrics (at a fixed training ratio).

    Call once per training ratio you want to highlight, e.g.:
        table_5pct = build_report_table(summary[summary.ratio == 0.05])
    """
    table = summary.copy()
    for m in METRIC_COLS:
        table[f"{m} (%)"] = (table[f"{m}_mean"] * 100).map("{:.2f}".format) + " ± " + (table[f"{m}_std"] * 100).map(
            "{:.2f}".format)
    return table[["ratio", "method"] + [f"{m} (%)" for m in METRIC_COLS]]


# ─── 7. Real-OOD Experiment ───────────────────────────────────────────────────

def run_real_ood(cubes, ood_cube_idx=-1, train_ratio=0.10, seed=42, train_max_pixels_per_class=None):
    """
    Train on all cubes except ood_cube_idx; test on the OOD cube.
    Returns DataFrame (rows = methods, cols = metrics).
    """
    print(f"\n[Skip] Skipping Real-OOD cross-cube experiment because the script is now in Intra-Cube mode.")
    return pd.DataFrame()


# ─── 8. Simulated OOD ─────────────────────────────────────────────────────────

def ood_illum(X, std=0.30, seed=1):
    rng = np.random.RandomState(seed)
    return X * (1.0 + std * rng.randn(X.shape[0], 1))


def ood_sensor_drift(X, sigma=0.10, seed=2):
    rng = np.random.RandomState(seed)
    return X + sigma * rng.randn(1, X.shape[1])


def ood_band_shift(X, shift_bands=2):
    return np.roll(X, shift_bands, axis=1)


def run_simulated_ood(cubes, train_ratio=0.10,
                      test_cube_idx=0, seed=42, train_max_pixels_per_class=None,
                      test_max_pixels_per_class=500):
    """
    Train on other cubes, apply simulated OOD to the test cube.
    Returns DataFrame (rows = methods, cols = ID + 3 OOD conditions).
    """
    X_test_cube, y_test_cube, test_name = cubes[test_cube_idx]

    train_cubes = [(X, y, name) for i, (X, y, name) in enumerate(cubes) if i != test_cube_idx]
    if not train_cubes:
        return pd.DataFrame()

    X_pool = np.vstack([c[0] for c in train_cubes])
    y_pool = np.concatenate([c[1] for c in train_cubes])

    n_cls = len(np.unique(y_pool))
    n_tr = max(int(train_ratio * len(y_pool)), n_cls)

    Xtr, _, ytr, _ = train_test_split(
        X_pool, y_pool, train_size=n_tr,
        stratify=y_pool, random_state=seed
    )

    Xtr, ytr = subsample_per_class(
        Xtr, ytr,
        max_pixels_per_class=train_max_pixels_per_class,
        seed=seed
    )

    X_test = X_test_cube
    y_test = y_test_cube

    # ── Strictly balance the test set to prevent Majority Class Collapse OA cheating ──
    X_test, y_test = subsample_per_class(
        X_test, y_test,
        max_pixels_per_class=test_max_pixels_per_class,
        seed=seed
    )

    # Subsample testing data if too large to prevent OOM
    max_test_pixels = 20000
    if len(y_test) > max_test_pixels:
        X_test, _, y_test, _ = train_test_split(
            X_test, y_test, train_size=max_test_pixels, stratify=y_test, random_state=seed
        )

    X_test = X_test.astype(np.float32, copy=False)

    conditions = {
        "ID": X_test,
        "OOD-Illum": ood_illum(X_test, std=0.35),
        "OOD-SensorDrift": ood_sensor_drift(X_test, sigma=0.12),
        "OOD-BandShift": ood_band_shift(X_test, shift_bands=2),
    }

    print(f"\nSimulated OOD on Intra-Cube  (test={test_name}, train_ratio={train_ratio:.0%})")
    records = {}
    for method_name, method_fn in METHODS.items():
        row = {}
        try:
            cond_names = list(conditions.keys())
            cond_X = list(conditions.values())
            all_Xte = np.vstack(cond_X)
            split_sizes = [len(X) for X in cond_X]

            if method_name == "F: PLS-DA":
                pls, classes = fit_plsda(Xtr, ytr)
                all_y_pred = predict_plsda(pls, classes, all_Xte)
                pred_splits = np.split(all_y_pred, np.cumsum(split_sizes)[:-1])
                for name, y_pred in zip(cond_names, pred_splits):
                    row[name] = float(accuracy_score(y_test, y_pred))
            else:
                is_snv_pre = method_name in SNV_PRE_METHODS
                xtr_arg = snv(Xtr).astype(np.float32, copy=False) if is_snv_pre else Xtr
                xte_arg = snv(all_Xte).astype(np.float32, copy=False) if is_snv_pre else all_Xte
                Xtr_f, all_Xte_f = method_fn(xtr_arg, xte_arg, ytr=ytr, pre_snv=is_snv_pre)
                skip_snv = method_name in METHODS_SKIP_POST_SNV
                if not skip_snv:
                    Xtr_f = snv(Xtr_f)
                    all_Xte_f = snv(all_Xte_f)
                sc = StandardScaler()
                clf = make_clf()
                clf.fit(sc.fit_transform(Xtr_f), ytr)
                te_splits = np.split(all_Xte_f, np.cumsum(split_sizes)[:-1])
                for name, Xte_f in zip(cond_names, te_splits):
                    row[name] = float(accuracy_score(y_test, clf.predict(sc.transform(Xte_f))))
        except Exception as e:
            row = {k: np.nan for k in conditions}
        records[method_name] = row
        print(f"  {method_name:<25s}  " +
              "  ".join(f"{k}={v:.3f}" for k, v in row.items()))

    return pd.DataFrame(records).T


# ─── 9. Plotting ──────────────────────────────────────────────────────────────

COLORS = {
    "A: Global TruncSVD": "#6366f1",
    "A*: Global TruncSVD (Centered)": "#4338ca",
    "A2: Global TruncSVD (SNV-Pre)": "#818cf8",
    "A2*: Global TruncSVD (SNV-Pre, Centered)": "#6366f1",
    "A3: Global TruncSVD (MSC-Pre)": "#a78bfa",
    "A3*: Global TruncSVD (MSC-Pre, Centered)": "#7c3aed",
    "A_drop1: Drop PC1 (PC2–PC20)": "#06b6d4",
    "A_L2: L2 row norm (no SNV)": "#0ea5e9",
    "A_drop1_scale: Drop PC1 + Scale (no SNV)": "#0284c7",
    "B2: Peak-Only(Raw)": "#22d3ee",
    "C2: Raw+Comp": "#fb923c",
    "D: Rect+Comp": "#ec4899",
    "E: Local+Comp": "#f59e0b",
}
STYLES = {
    "A: Global TruncSVD": "--",
    "A*: Global TruncSVD (Centered)": "--",
    "A2: Global TruncSVD (SNV-Pre)": ":",
    "A2*: Global TruncSVD (SNV-Pre, Centered)": ":",
    "A3: Global TruncSVD (MSC-Pre)": "-.",
    "A3*: Global TruncSVD (MSC-Pre, Centered)": "-.",
    "A_drop1: Drop PC1 (PC2–PC20)": "-",
    "A_L2: L2 row norm (no SNV)": "-",
    "A_drop1_scale: Drop PC1 + Scale (no SNV)": "-",
    "B2: Peak-Only(Raw)": "-.",
    "C2: Raw+Comp": "-.",
    "D: Rect+Comp": ":",
    "E: Local+Comp": ":",
}


def plot_loco_curves(summary, metric="OA", out_dir="."):
    """
    Learning curves from Intra-cube CV: mean ± std band per method.
    summary: output of summarize_loco().
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    ratios_pct = sorted(summary["ratio"].unique())
    ratios_pct_labels = [r * 100 for r in ratios_pct]

    for method_name in METHODS:
        sub = summary[summary["method"] == method_name].sort_values("ratio")
        mean = sub[f"{metric}_mean"].values * 100
        std = sub[f"{metric}_std"].values * 100
        ax.plot(ratios_pct_labels, mean,
                color=COLORS.get(method_name, "gray"),
                linestyle=STYLES.get(method_name, "-"),
                linewidth=2.2, marker="o", markersize=6,
                label=method_name)
        ax.fill_between(ratios_pct_labels,
                        mean - std, mean + std,
                        alpha=0.12,
                        color=COLORS.get(method_name, "gray"))

    ax.set_xlabel("Training ratio (%)", fontsize=12)
    ax.set_ylabel(f"{metric} (%)", fontsize=12)
    ax.set_title(f"Intra-Cube learning curves — {metric} (mean ± std)", fontsize=13)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    out = Path(out_dir) / f"loco_curves_{metric.lower()}.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Saved: {out}")


def plot_metrics_heatmap(summary, ratio=0.05, out_dir="."):
    """
    Heatmap: methods × metrics at a given training ratio.
    Useful for the main paper table figure.
    """
    sub = summary[summary["ratio"] == ratio].set_index("method")
    mean_cols = [f"{m}_mean" for m in METRIC_COLS]
    data = sub[mean_cols].rename(columns={f"{m}_mean": m for m in METRIC_COLS})

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(data.values, aspect="auto", cmap="YlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(METRIC_COLS)))
    ax.set_xticklabels(METRIC_COLS, fontsize=11)
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data.index.tolist(), fontsize=9)
    for i in range(len(data)):
        for j, m in enumerate(METRIC_COLS):
            val = data.iloc[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=9, color="black" if val < 0.7 else "white")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"Method × Metric heatmap  (training ratio = {ratio:.0%})",
                 fontsize=12)
    fig.tight_layout()
    out = Path(out_dir) / f"metrics_heatmap_{int(ratio * 100)}pct.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Saved: {out}")


def plot_ood_bar(df_ood, out_dir="."):
    """
    Grouped bar chart: methods × OOD conditions.
    df_ood: output of run_real_ood() or run_simulated_ood().
    """
    conds = df_ood.columns.tolist()
    x = np.arange(len(conds))
    width = 0.10
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (method_name, row) in enumerate(df_ood.iterrows()):
        vals = row[conds].values.astype(float) * 100
        offset = (i - len(METHODS) / 2) * width
        ax.bar(x + offset, vals, width,
               color=COLORS.get(method_name, "gray"), alpha=0.85,
               label=method_name)
    ax.set_xticks(x)
    ax.set_xticklabels(conds, fontsize=10)
    ax.set_ylabel("Overall Accuracy (%)", fontsize=12)
    ax.set_title("OOD robustness (10 % training)", fontsize=13)
    ax.legend(fontsize=8, loc="upper right", ncol=2)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    out = Path(out_dir) / "ood_bar.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Saved: {out}")


def plot_band_coverage(out_dir="."):
    """Visualise prior vs complement band allocation."""
    comp = complement_bands(len(WAVELENGTHS), PRIOR_PEAKS_IDX)
    pmask = prior_band_mask(len(WAVELENGTHS), PRIOR_PEAKS_IDX)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.scatter(WAVELENGTHS[pmask], np.ones(pmask.sum()),
               c="#10b981", s=20, alpha=0.8, label="Prior region (±3σ)")
    ax.scatter(WAVELENGTHS[comp], np.zeros(len(comp)),
               c="#f97316", s=20, alpha=0.8, label="Complement")
    for i, c in enumerate(PRIOR_PEAKS_IDX):
        ax.axvline(WAVELENGTHS[c], color="#10b981", alpha=0.4, linewidth=1.2)
        ax.text(WAVELENGTHS[c], 1.10, f"{PRIOR_PEAKS_NM[i]}nm",
                ha="center", va="bottom", fontsize=8, color="#10b981")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Complement", "Prior"], fontsize=10)
    ax.set_xlabel("Wavelength (nm)", fontsize=11)
    ax.set_title("Band allocation: prior (green) vs complement (orange)")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    out = Path(out_dir) / "band_coverage.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Saved: {out}")


# ─── 10. Main ─────────────────────────────────────────────────────────────────

def main_swir(data_dir="/kaggle/input/datasets/moranyin/swir-cubes/swir_cubes",
              mask_dir=None,
              cube_key="cubo",
              mask_key=None,
              wvl_key=None,
              exclude_bg_label=9,
              train_max_pixels_per_class=None,
              test_max_pixels_per_class=500,
              ood_cube_idx=None,
              out_dir="/kaggle/working/output",
              csv_path=None,
              use_synthetic=False):
    """
    Full pipeline for SWIR Plastics LOCO-CV experiment.

    Parameters
    ----------
    data_dir             : directory containing cube*.mat files
    cube_key             : variable name of hypercube inside .mat
    mask_key             : variable name of label mask inside .mat
    wvl_key              : variable name of wavelength vector (or None)
    exclude_bg_label     : mask value to treat as background
    max_pixels_per_class : cap pixel count per class per cube (avoids RAM issues)
    ood_cube_idx         : index of the dedicated OOD cube (fixed hold-out)
                           set to None to run full LOCO without a fixed OOD cube
    out_dir              : output directory for CSVs and figures
    use_synthetic        : True → skip real data, use synthetic (sanity-check only)

    Quick-start
    -----------
    1. Put all cube*.mat files into a folder, e.g. "swir_cubes/"
    2. Run:  python prior_guided_pca.py
       or:   from prior_guided_pca import main; main(data_dir="swir_cubes")
    3. If unsure of variable names inside .mat:
           from prior_guided_pca import probe_mat_keys
           probe_mat_keys("swir_cubes/cube_Jan20.mat")
       Then set cube_key / mask_key accordingly.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    if not use_synthetic:
        if csv_path is not None:
            csv_path = Path(csv_path)
            if not csv_path.exists():
                raise FileNotFoundError(f"CSV not found: {csv_path}")
        else:
            script_path = globals().get("__file__", None)
            base_dir = Path(script_path).parent if script_path else Path.cwd()
            csv_path = base_dir / "spectradictionary.csv"

        if csv_path.exists():
            ratios = [0.01, 0.05, 0.10,0.15, 0.20,0.30,0.50]
            seeds = [42, 43, 44, 45, 46]
            df_csv, df_pc = run_csv_experiment_kfold(
                csv_path=csv_path,
                ratios=ratios,
                n_splits=5,
                seeds=seeds,
                water_filter=0,
                band_range=(940, 1680),
                quiet=False,
            )
            df_csv.to_csv(out_dir / "csv_results_raw.csv", index=False)
            print(f"\nRaw CSV results saved to {out_dir / 'csv_results_raw.csv'}")

            summary_csv = summarize_loco(df_csv)
            summary_csv.to_csv(out_dir / "csv_results_summary.csv", index=False)
            print(f"CSV summary saved to {out_dir / 'csv_results_summary.csv'}")

            # Save PC1/PC2 summary (mean over seeds and folds)
            if not df_pc.empty:
                pc_summary = df_pc.groupby(['ratio', 'method'])[['PC1', 'PC2']].mean().reset_index()
                pc_summary.to_csv(out_dir / "pc_variance_summary.csv", index=False)
                print(f"PC variance summary saved to {out_dir / 'pc_variance_summary.csv'}")

            if 0.05 in summary_csv["ratio"].values:
                print("\n── Table: metrics at 5 % training ratio (mean ± std over 5 folds) ──")
                tbl = build_report_table(summary_csv[summary_csv.ratio == 0.05])
                print(tbl.to_string(index=False))

            print("\n── Exp OOD: water conditions ──")
            ood_ratios = [0.01, 0.05, 0.10,0.15,0.20,0.30,0.50]
            ood_seeds = [42, 43, 44, 45, 46]
            df_ood = run_water_ood_experiment(
                csv_path=csv_path,
                ratios=ood_ratios,
                seeds=ood_seeds,
                min_plastic_frac=0.0,
                band_range=(940, 1680),
                sigma_nm=7.0,
                train_water=0,
                id_test_size=0.2,
                test_max_per_class=500
            )
            # Save OOD summary (mean + std over seeds and folds)
            ood_cols = ["ID", "OOD-Turbid", "OOD-Foamy",
                        "ID-IllumShift", "ID-SensorDrift", "ID-BandShift",
                        "OOD-IllumShift", "OOD-SensorDrift", "OOD-BandShift"]
            ood_summary = df_ood.groupby(['ratio', 'method'])[ood_cols].agg(['mean', 'std'])
            ood_summary.columns = [f'{col}_{stat}' for col, stat in ood_summary.columns]
            ood_summary = ood_summary.reset_index()
            ood_csv = out_dir / "ood_water_results_summary.csv"
            ood_summary.to_csv(ood_csv, index=False)
            print(f"Saved: {ood_csv}")
            for r in ood_ratios:
                plot_ood_heatmap(ood_summary, ratio=r, out_dir=out_dir)
            print("\n── OOD Macro-F1 (mean) at ratio=100% ──")
            f1_cols = [c for c in ood_summary.columns if c.endswith("_F1_mean")]
            f1_sub = ood_summary[ood_summary["ratio"] == 1.0]
            if not f1_sub.empty and f1_cols:
                f1_tbl = f1_sub.set_index("method")[f1_cols].copy()
                f1_tbl.columns = [c.replace("_F1_mean", "") for c in f1_tbl.columns]
                print(f1_tbl.to_string(float_format=lambda x: f"{x:.4f}"))
            print("\nDone. Output files in:", out_dir)
            return

    # ── Load data ──
    if use_synthetic:
        print("Generating synthetic data (sanity-check only) …")
        X_syn, y_syn = make_synthetic_data()
        cubes = [(X_syn, y_syn, "synthetic")]
        cube_infos = None
    else:
        print(f"Discovering SWIR cubes from: {data_dir}")
        try:
            cube_infos = discover_swir_cubes(
                data_dir=data_dir,
                mask_dir=mask_dir,
                mat_files=None
            )
        except FileNotFoundError as e:
            raise FileNotFoundError(
                f"{e}\n\n"
                "This run is using the legacy SWIR .mat pipeline, but no cube*.mat files were found.\n"
                "If you only have the CSV library, run:\n"
                "  main(csv_path='/kaggle/input/<dataset>/spectradictionary.csv')\n"
                "and make sure that file exists.\n"
                "Alternatively, set use_synthetic=True for a synthetic sanity-check."
            ) from e
        if not cube_infos:
            raise RuntimeError("No cube metadata found — check data_dir and mask_dir.")

    # ── Exp 1: Intra-Cube Evaluation ──
    print("── Exp 1: Intra-Cube CV (Train/Test within the same cube) ──")
    if use_synthetic:
        n_classes = len(np.unique(y_syn))
        print(f"\nTotal cubes: 1 Global classes: {n_classes}")
        print(f"Prior peaks: {list(zip(PRIOR_PEAKS_NM, PRIOR_PEAKS_IDX))}")
        print(f"σ = {SIGMA_BANDS} bands (3σ = ±{3 * SIGMA_BANDS} bands)\n")

        plot_band_coverage(out_dir)

        print("── Exp 1: synthetic train/test split ──")
        X, y, name = cubes[0]
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                              stratify=y, random_state=42)
        records = []
        for ratio in TRAIN_RATIOS:
            n_tr = max(int(ratio * len(ytr)), n_classes)
            Xs, _, ys, _ = train_test_split(Xtr, ytr,
                                            train_size=n_tr,
                                            stratify=ytr,
                                            random_state=42)
            for mname, mfn in METHODS.items():
                if mname == "F: PLS-DA":
                    pls, classes = fit_plsda(Xs, ys)
                    y_pred = predict_plsda(pls, classes, Xte)
                    metrics = compute_metrics(yte, y_pred)
                else:
                    Xtr_f, Xte_f = mfn(Xs, Xte, ytr=ys)
                    metrics = evaluate(Xtr_f, Xte_f, ys, yte)
                records.append({
                    "fold": 0,
                    "cube_test": name,
                    "ratio": ratio,
                    "n_train": n_tr,
                    "method": mname,
                    **metrics
                })
        df_loco = pd.DataFrame(records)

    else:
        print(f"\nTotal cubes discovered: {len(cube_infos)}")
        print(f"Prior peaks: {list(zip(PRIOR_PEAKS_NM, PRIOR_PEAKS_IDX))}")
        print(f"σ = {SIGMA_BANDS} bands (3σ = ±{3 * SIGMA_BANDS} bands)\n")

        plot_band_coverage(out_dir)

        print("── Exp 1: Intra-cube CV ──")
        df_loco = run_intra_cube_cv(
            cube_infos,
            ratios=TRAIN_RATIOS,
            seed=42,
            train_max_pixels_per_class=train_max_pixels_per_class,
            test_max_pixels_per_class=test_max_pixels_per_class,
            max_pixels_per_cube=100000,
            cube_key=cube_key,
            mask_key=mask_key,
            wvl_key=wvl_key,
            exclude_bg_label=exclude_bg_label
        )

    df_loco.to_csv(out_dir / "intra_cube_raw.csv", index=False)
    print(f"\nRaw Intra-cube results saved to {out_dir}/intra_cube_raw.csv")

    summary = summarize_loco(df_loco)
    summary.to_csv(out_dir / "intra_cube_summary.csv", index=False)
    print(f"Intra-cube summary saved to {out_dir}/intra_cube_summary.csv")

    # Print publication table at 5 % training ratio
    print("\n── Table: metrics at 5 % training ratio (mean ± std over folds) ──")
    tbl = build_report_table(summary[summary.ratio == 0.05])
    print(tbl.to_string(index=False))

    # Learning curve plots for all four metrics
    for metric in METRIC_COLS:
        plot_loco_curves(summary, metric=metric, out_dir=out_dir)

    # Heatmap at representative ratios
    for ratio in [0.01, 0.02, 0.05, 0.10, 0.20,0.30]:
        if ratio in summary["ratio"].values:
            plot_metrics_heatmap(summary, ratio=ratio, out_dir=out_dir)

    # ── Exp 2 & 3 need full cubes in memory ──
    if not use_synthetic:
        print("\n── Loading full cubes for OOD Experiments ──")
        cubes = load_swir_plastics(
            data_dir=data_dir,
            mask_dir=mask_dir,
            mat_files=None,
            cube_key=cube_key,
            mask_key=mask_key,
            wvl_key=wvl_key,
            exclude_bg_label=exclude_bg_label
        )

    # ── Exp 2: Real-OOD (held-out cube) ──
    if len(cubes) >= 3 and ood_cube_idx is not None:
        print(f"\n── Exp 2: Real-OOD (cube {cubes[ood_cube_idx][2]}) ──")
        df_ood = run_real_ood(cubes, ood_cube_idx=ood_cube_idx,
                              train_max_pixels_per_class=train_max_pixels_per_class)
        # We skip plotting since Real-OOD is disabled in intra-cube mode

    # ── Exp 3: Simulated OOD ──
    print("\n── Exp 3: Simulated OOD (illumination / sensor drift) ──")
    sim_test_idx = next((i for i, (_, _, n) in enumerate(cubes) if n == "cube_7Feb24b"), 0)
    df_sim_ood = run_simulated_ood(cubes, test_cube_idx=sim_test_idx,
                                   train_max_pixels_per_class=train_max_pixels_per_class,
                                   test_max_pixels_per_class=test_max_pixels_per_class)
    ood_csv = out_dir / "ood_simulated.csv"
    df_sim_ood.to_csv(ood_csv)
    print(f"Saved: {ood_csv}")
    plot_ood_bar(df_sim_ood, out_dir=out_dir)

    print("\nDone. Output files in:", out_dir)


def main(out_dir="/kaggle/working/output",
         csv_path="/kaggle/input/datasets/moranyin/swir-cubes1/spectradictionary.csv",
         use_synthetic=False):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    if use_synthetic:
        X_syn, y_syn = make_synthetic_data()
        n_classes = len(np.unique(y_syn))
        Xtr, Xte, ytr, yte = train_test_split(X_syn, y_syn, test_size=0.3,
                                              stratify=y_syn, random_state=42)
        records = []
        for ratio in TRAIN_RATIOS:
            n_tr = max(int(ratio * len(ytr)), n_classes)
            Xs, _, ys, _ = train_test_split(Xtr, ytr,
                                            train_size=n_tr,
                                            stratify=ytr,
                                            random_state=42)
            for mname, mfn in METHODS.items():
                if mname == "F: PLS-DA":
                    pls, classes = fit_plsda(Xs, ys)
                    y_pred = predict_plsda(pls, classes, Xte)
                    metrics = compute_metrics(yte, y_pred)
                else:
                    Xtr_f, Xte_f = mfn(Xs, Xte, ytr=ys)
                    metrics = evaluate(Xtr_f, Xte_f, ys, yte)
                records.append({
                    "fold": 0,
                    "cube_test": "synthetic",
                    "ratio": ratio,
                    "n_train": n_tr,
                    "method": mname,
                    **metrics
                })
        df_csv = pd.DataFrame(records)
        df_csv.to_csv(out_dir / "synthetic_results_raw.csv", index=False)
        print(f"Saved: {out_dir / 'synthetic_results_raw.csv'}")
        summary = summarize_loco(df_csv)
        summary.to_csv(out_dir / "synthetic_results_summary.csv", index=False)
        print(f"Saved: {out_dir / 'synthetic_results_summary.csv'}")
        print("\nDone. Output files in:", out_dir)
        return

    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    ratios = [ 0.01, 0.10,0.15, 0.20,0.30,0.50,1.00]
    seeds = [42, 43, 44, 45, 46]
    df_csv, df_pc = run_csv_experiment_kfold(
        csv_path=csv_path,
        ratios=ratios,
        n_splits=5,
        seeds=seeds,
        water_filter=0,
        band_range=(940, 1680),
        quiet=False,
    )
    df_csv.to_csv(out_dir / "csv_results_raw.csv", index=False)
    print(f"\nRaw CSV results saved to {out_dir / 'csv_results_raw.csv'}")

    summary_csv = summarize_loco(df_csv)
    summary_csv.to_csv(out_dir / "csv_results_summary.csv", index=False)
    print(f"CSV summary saved to {out_dir / 'csv_results_summary.csv'}")

    # Save PC1/PC2 summary (mean over seeds and folds)
    if not df_pc.empty:
        pc_summary = df_pc.groupby(['ratio', 'method'])[['PC1', 'PC2']].mean().reset_index()
        pc_summary.to_csv(out_dir / "pc_variance_summary.csv", index=False)
        print(f"PC variance summary saved to {out_dir / 'pc_variance_summary.csv'}")

    if 0.05 in summary_csv["ratio"].values:
        print("\n── Table: metrics at 5 % training ratio (mean ± std over 5 folds) ──")
        tbl = build_report_table(summary_csv[summary_csv.ratio == 0.05])
        print(tbl.to_string(index=False))

    print("\n── Exp OOD: water conditions ──")
    ood_ratios = [0.01, 0.10,0.15, 0.20,0.30,0.50,1.00]
    ood_seeds = [42, 43, 44, 45, 46]
    df_ood = run_water_ood_experiment(
        csv_path=csv_path,
        ratios=ood_ratios,
        seeds=ood_seeds,
        min_plastic_frac=0.0,
        band_range=(940, 1680),
        sigma_nm=7.0,
        train_water=0,
        id_test_size=0.2,
        test_max_per_class=500,
        use_multi_train=True
    )
    # Save OOD summary (mean + std over seeds and folds)
    ood_cols = ["ID", "OOD-Turbid", "OOD-Foamy",
                "ID-IllumShift", "ID-SensorDrift", "ID-BandShift",
                "OOD-IllumShift", "OOD-SensorDrift", "OOD-BandShift"]
    ood_summary = df_ood.groupby(['ratio', 'method'])[ood_cols].agg(['mean', 'std'])
    ood_summary.columns = [f'{col}_{stat}' for col, stat in ood_summary.columns]
    ood_summary = ood_summary.reset_index()
    ood_csv = out_dir / "ood_water_results_summary.csv"
    ood_summary.to_csv(ood_csv, index=False)
    print(f"Saved: {ood_csv}")
    for r in ood_ratios:
        plot_ood_heatmap(ood_summary, ratio=r, out_dir=out_dir)
    print("\n── OOD Macro-F1 (mean) at ratio=100% ──")
    f1_cols = [c for c in ood_summary.columns if c.endswith("_F1_mean")]
    f1_sub = ood_summary[ood_summary["ratio"] == 1.0]
    if not f1_sub.empty and f1_cols:
        f1_tbl = f1_sub.set_index("method")[f1_cols].copy()
        f1_tbl.columns = [c.replace("_F1_mean", "") for c in f1_tbl.columns]
        print(f1_tbl.to_string(float_format=lambda x: f"{x:.4f}"))

    print("\nDone. Output files in:", out_dir)

In [ ]:

out1 = "/kaggle/working/output_with_snv"
main(out_dir=out1, csv_path="/kaggle/input/datasets/moranyin/swir-cubes1/spectradictionary.csv", use_synthetic=False)

snv_saved = snv
def snv(X, eps=1e-8):
    return X

out2 = "/kaggle/working/output_no_snv"
main(out_dir=out2, csv_path="/kaggle/input/datasets/moranyin/swir-cubes1/spectradictionary.csv", use_synthetic=False)


[CSV] loaded 193 samples | 6 materials | 741 bands (940–1680 nm)
  PET: 54 samples
  HDPE: 30 samples
  LDPE: 18 samples
  PP: 52 samples
  EPSF: 16 samples
  Weathered: 23 samples
Prior peaks: [(1130, 190), (1210, 270), (1390, 450), (1490, 550), (1660, 720)]
σ = 7 bands

Seed 1/5 (seed=42)

── Seed 1/5 Fold 1/5  train=154  test=39  classes=6 ──
  [A: Global PCA] PC1: 98.78%, PC2: 1.01%
  [A*: Centered] PC1: 97.23%, PC2: 2.48%
  [A2: SNV-Pre] PC1: 81.31%, PC2: 8.95%
  [A2*: SNV-Pre Centered] PC1: 48.30%, PC2: 31.24%
  [A3: MSC-Pre] PC1: 98.30%, PC2: 0.94%
  [A3*: MSC-Pre Centered] PC1: 55.26%, PC2: 28.61%
  [] PC1: 98.90%, PC2: 0.99%
  [] PC1: 98.90%, PC2: 0.99%
  ⚠ E: Local+Comp failed: unsupported format string passed to NoneType.__format__
  [] PC1: 98.90%, PC2: 0.99%
  ratio=1.0% n_train=    6  GlobalPCA OA=0.359  Raw+Comp OA=0.667
  [A: Global PCA] PC1: 98.68%, PC2: 1.09%
  [A*: Centered] PC1: 97.86%, PC2: 1.67%
  [A2: SNV-Pre] PC1: 75.34%, PC2: 14.16%
  [A2*: SNV-Pre Centered] PC